# 5-2: LiP-MS Structure Mapping

Maps significant LiP-MS peptides and metabolite-correlating peptides onto a 3D protein structure.

Supports **RCSB PDB** structures and **AlphaFold EBI** predicted structures.

**Sources:**
- LiP-MS hits: `results/diffStructurome_G/N/P.tsv`
- Correlations: `source_data/LiPMS_vs_Metab.tsv`
- Structure: configured in the Parameters cell below


In [130]:
import pandas as pd
import numpy as np
import requests
import os
import pickle
import py3Dmol
import freesasa
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
from plotnine import *

print(f'py3Dmol version: {py3Dmol.__version__}')
print(f'Working directory: {os.getcwd()}')

py3Dmol version: 2.5.4
Working directory: /Users/xhernandez/Documents/pmi_translation


# SES1

## Parameters

Edit the cell below to re-run this analysis for a different protein complex. All downstream cells adapt automatically.

In [174]:
# ============================================================
# PARAMETERS — edit only this cell to switch to another structure
# ============================================================

# Structure source: 'pdb' downloads from RCSB, 'alphafold' downloads from AlphaFold EBI
STRUCTURE_SOURCE = 'alphafold'

# RCSB PDB identifier (used when STRUCTURE_SOURCE = 'pdb')
PDB_ID           = None

# UniProt accession for AlphaFold (used when STRUCTURE_SOURCE = 'alphafold')
# e.g. 'P38179' — downloads AF-P38179-F1-model_v4.cif from alphafold.ebi.ac.uk
ALPHAFOLD_ID     = 'P07284'

ANALYSIS_LABEL   = 'SES1'   # label used in plot titles
OUTPUT_PREFIX    = 'SES1'   # prefix for all output files

# Restrict to a subset of UniProt IDs present in both the LiP-MS data and the structure.
# None = use ALL proteins found in the intersection automatically.
# For AlphaFold, leave as None (auto) or set to [ALPHAFOLD_ID].
UNIPROT_FILTER = None

# Color palette (used throughout downstream cells)
COLOR_FILTERED   = '#888888'  # chains NOT in UNIPROT_FILTER (if set); use None to hide them
COLOR_BASE       = '#888888'  # focus protein chains (all proteins if UNIPROT_FILTER is None)
COLOR_SIG        = '#ea9c8e'  # significant LiP peptides, NOT metabolite-correlated
COLOR_CORR       = '#d8624d'  # significant LiP peptides, correlating with metabolites


## 1. Load LiP-MS Data

In [175]:
# Load differential structurome results from all three nutrient conditions
diff_g = pd.read_csv('results/diffStructurome_G.tsv', sep='\t', index_col='quantID')
diff_n = pd.read_csv('results/diffStructurome_N.tsv', sep='\t', index_col='quantID')
diff_p = pd.read_csv('results/diffStructurome_P.tsv', sep='\t', index_col='quantID')

# Build annotprot: union of all conditions, deduplicated by quantID
annot_cols = ['Peptide', 'Protein', 'gene_name', 'startPosition', 'endPosition', 'isTryptic']
annotprot = pd.concat([diff_g[annot_cols], diff_n[annot_cols], diff_p[annot_cols]])
annotprot = annotprot[~annotprot.index.duplicated(keep='first')]

# siglip: peptides significant (adj.P.Val < 0.05) in at least one condition
siglip_g = set(diff_g.index[(diff_g['adj.P.Val'] < 0.05) & diff_g['adj.P.Val'].notna()])
siglip_n = set(diff_n.index[(diff_n['adj.P.Val'] < 0.05) & diff_n['adj.P.Val'].notna()])
siglip_p = set(diff_p.index[(diff_p['adj.P.Val'] < 0.05) & diff_p['adj.P.Val'].notna()])
siglip = list(siglip_g | siglip_n | siglip_p)

print(f'Total peptides in annotprot: {len(annotprot)}')
print(f'Significant peptides (siglip): {len(siglip)}')
print(f'  G: {len(siglip_g)}, N: {len(siglip_n)}, P: {len(siglip_p)}')
annotprot.head(3)

Total peptides in annotprot: 29732
Significant peptides (siglip): 5540
  G: 3204, N: 1112, P: 1782


,Peptide,Protein,gene_name,startPosition,endPosition,isTryptic
quantID,,,,,,
EMIISSLSPLEPHSQNEILQYYL,EMIISSLSPLEPHSQNEILQYYL,P12945,NAT1,832,854,Specific
SFDDLHK,SFDDLHK,P17709,GLK1,2,8,Specific-N
EQNEDLGLSDIKVE,EQNEDLGLSDIKVE,P53303,ZPR1,473,486,Specific


## 2. Load Correlation Data

In [176]:
# Load LiP-MS vs Metabolomics correlation results
# X1 = peptide (quantID), X2 = metabolite, lfdr = local false discovery rate
cors = pd.read_csv('source_data/LiPMS_vs_Metab.tsv', sep='\t', index_col=0)
print(f'Columns: {list(cors.columns)}')
print(f'Total peptide-metabolite pairs tested: {len(cors)}')

# Correlating peptides: those with lfdr < 0.05 in at least one metabolite pair
correlating_pepts = set(cors.loc[cors['lfdr'] < 0.05, 'X1'])
print(f'Peptides with significant metabolite correlations (lfdr < 0.05): {len(correlating_pepts)}')

# Classify siglip peptides into two categories:
#   sig_only     → significant LiP but NOT metabolite-correlated → color #4c7b6a
#   sig_and_corr → significant LiP AND metabolite-correlated   → color #d8624d
sig_only     = [p for p in siglip if p not in correlating_pepts]
sig_and_corr = [p for p in siglip if p in correlating_pepts]

print(f'\nSignificant, non-correlating (→ #4c7b6a): {len(sig_only)}')
print(f'Significant AND correlating  (→ #d8624d): {len(sig_and_corr)}')

Columns: ['X1', 'X2', 'pval', 'qval', 'lfdr', 'R2', 'annot_pept', 'annot_metab', 'GO_translation']
Total peptide-metabolite pairs tested: 1429866
Peptides with significant metabolite correlations (lfdr < 0.05): 1558

Significant, non-correlating (→ #4c7b6a): 3982
Significant AND correlating  (→ #d8624d): 1558


## 3. Fetch Structure and SIFTS Mapping


In [177]:
os.makedirs('data', exist_ok=True)

if STRUCTURE_SOURCE == 'alphafold':
    struct_id = f'AF-{ALPHAFOLD_ID}-F1'
    cif_path  = f'data/{struct_id}.cif'
    if not os.path.exists(cif_path):
        print(f'Downloading AlphaFold structure for {ALPHAFOLD_ID}...')
        url  = f'https://alphafold.ebi.ac.uk/files/{struct_id}-model_v6.cif'
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cif_path, 'w') as f:
            f.write(resp.text)
        print(f'Saved to {cif_path}')
    else:
        print(f'Using cached {cif_path}')
else:
    struct_id = PDB_ID
    cif_path  = f'data/{PDB_ID}.cif'
    if not os.path.exists(cif_path):
        print(f'Downloading {PDB_ID} from RCSB (may take a minute for large structures)...')
        url  = f'https://files.rcsb.org/download/{PDB_ID}.cif'
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cif_path, 'w') as f:
            f.write(resp.text)
        print(f'Saved to {cif_path}')
    else:
        print(f'Using cached {cif_path}')

file_size_mb = os.path.getsize(cif_path) / 1e6
print(f'File size: {file_size_mb:.1f} MB')


Using cached data/AF-P07284-F1.cif
File size: 0.4 MB


In [178]:
def get_sifts_mapping(pdb_id):
    """
    Fetch UniProt -> PDB chain/residue mapping from PDBe SIFTS API.
    Returns:
        sifts:      {uniprot_id: [{'chain_id', 'unp_start', 'unp_end', 'pdb_start', 'pdb_end'}]}
        sifts_ident: {uniprot_id: identifier_string}  e.g. 'RL2A_YEAST'
    """
    url = f'https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id.lower()}'
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    pdb_key = pdb_id.lower()

    sifts       = {}
    sifts_ident = {}
    for uniprot_id, info in data[pdb_key]['UniProt'].items():
        chain_maps = []
        for m in info['mappings']:
            chain_maps.append({
                'chain_id': m['chain_id'],
                'unp_start': m['unp_start'],
                'unp_end':   m['unp_end'],
                'pdb_start': m['start']['residue_number'],
                'pdb_end':   m['end']['residue_number'],
            })
        sifts[uniprot_id]       = chain_maps
        sifts_ident[uniprot_id] = info.get('identifier', '')
    return sifts, sifts_ident


if STRUCTURE_SOURCE == 'alphafold':
    # AlphaFold models always use chain 'A' with residue numbers = UniProt positions.
    # Fetch exact sequence length from UniProt REST API.
    print(f'AlphaFold mode: fetching sequence length for {ALPHAFOLD_ID} from UniProt...')
    try:
        _resp = requests.get(
            f'https://rest.uniprot.org/uniprotkb/{ALPHAFOLD_ID}.json', timeout=30)
        _resp.raise_for_status()
        _seq_len = _resp.json()['sequence']['length']
        print(f'  Sequence length: {_seq_len} aa')
    except Exception as e:
        print(f'  UniProt API failed ({e}), falling back to length=10000')
        _seq_len = 10000
    sifts = {
        ALPHAFOLD_ID: [{'chain_id': 'A',
                         'unp_start': 1,    'unp_end': _seq_len,
                         'pdb_start': 1,    'pdb_end': _seq_len}]
    }
    sifts_ident = {ALPHAFOLD_ID: ALPHAFOLD_ID}
    print(f'Synthetic SIFTS: chain A, residues 1-{_seq_len} -> {ALPHAFOLD_ID}')
else:
    print(f'Fetching SIFTS mapping for {PDB_ID}...')
    sifts, sifts_ident = get_sifts_mapping(PDB_ID)
    print(f'Proteins with SIFTS mappings in {PDB_ID}: {len(sifts)}')
    print(f'Example UniProt IDs: {list(sifts.keys())[:8]}')


AlphaFold mode: fetching sequence length for P07284 from UniProt...
  Sequence length: 462 aa
Synthetic SIFTS: chain A, residues 1-462 -> P07284


## 4. Identify Peptides

In [179]:
# Find which LiP-MS proteins overlap with proteins in the ribosome structure
# Handle comma-separated protein IDs (ambiguously assigned peptides)
def _proteins_of(protein_str):
    """Return list of individual UniProt IDs from a possibly comma-separated string."""
    return [p.strip() for p in str(protein_str).split(',')]

lipm_proteins     = {p for prot_str in annotprot['Protein'] for p in _proteins_of(prot_str)}
ribosome_proteins = set(sifts.keys())
matched_proteins  = lipm_proteins & ribosome_proteins

# Apply optional UniProt filter (set UNIPROT_FILTER in the Parameters cell)
if UNIPROT_FILTER is not None:
    matched_proteins = matched_proteins & set(UNIPROT_FILTER)
    print(f'Restricted to {len(UNIPROT_FILTER)} specified UniProt IDs \u2192 {len(matched_proteins)} matched')

print(f'Proteins in LiP-MS dataset:  {len(lipm_proteins)}')
print(f'Proteins in {PDB_ID}:         {len(ribosome_proteins)}')
print(f'Matched proteins:            {len(matched_proteins)}')

# Filter peptide lists to matched proteins only
def filter_to_target(pept_list):
    return [p for p in pept_list
            if p in annotprot.index and
            any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]

siglip_ribo       = filter_to_target(siglip)
sig_only_ribo     = filter_to_target(sig_only)
sig_and_corr_ribo = filter_to_target(sig_and_corr)

print(f'\nPeptides from matched proteins in siglip:      {len(siglip_ribo)}')
print(f'  Sig, non-correlating (\u2192 {COLOR_SIG}):  {len(sig_only_ribo)}')
print(f'  Sig AND correlating  (\u2192 {COLOR_CORR}):  {len(sig_and_corr_ribo)}')

# All peptides from matched proteins (background for SASA analysis)
all_ribo_pepts = [p for p in annotprot.index
                  if any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]
print(f'\nAll matched-protein peptides (SASA background): {len(all_ribo_pepts)}')

# Show matched proteins and gene names
ribo_annot = (annotprot[annotprot['Protein'].apply(
                  lambda x: any(p in matched_proteins for p in _proteins_of(x)))]
              [['Protein', 'gene_name']].drop_duplicates().sort_values('gene_name'))
print(f'\nMatched proteins ({len(ribo_annot)}):')
display(ribo_annot)


Proteins in LiP-MS dataset:  2510
Proteins in None:         1
Matched proteins:            1

Peptides from matched proteins in siglip:      5
  Sig, non-correlating (→ #ea9c8e):  2
  Sig AND correlating  (→ #d8624d):  3

All matched-protein peptides (SASA background): 36

Matched proteins (1):


,Protein,gene_name
quantID,,
ELVSCSNCTDYQSR,P07284,SES1


## 5. Map Peptide Positions to PDB Residues

Uses SIFTS to convert UniProt sequence positions (startPosition, endPosition from LiP-MS) to PDB chain + residue numbers.

In [180]:
from collections import defaultdict

def map_peptide_to_pdb(pept_start, pept_end, chain_maps):
    """
    Map UniProt peptide [pept_start, pept_end] to PDB chain + residue list.
    Handles cases where a peptide spans multiple mapping segments.
    Returns: [{'chain': chain_id, 'residues': [res1, res2, ...]}]
    """
    results = []
    for cm in chain_maps:
        ov_start = max(pept_start, cm['unp_start'])
        ov_end   = min(pept_end,   cm['unp_end'])
        if ov_start <= ov_end:
            offset  = cm['pdb_start'] - cm['unp_start']
            pdb_res = list(range(ov_start + offset, ov_end + offset + 1))
            results.append({'chain': cm['chain_id'], 'residues': pdb_res})
    return results

def map_pept_list(pept_list):
    """Map a list of peptide IDs to PDB chain + residue positions.
    Handles peptides ambiguously assigned to multiple comma-separated proteins:
    tries each protein with its corresponding start/end position in order,
    and uses the first one that maps successfully to the PDB structure."""
    result = {}
    for pept_id in pept_list:
        if pept_id not in annotprot.index:
            continue
        row      = annotprot.loc[pept_id]
        proteins = _proteins_of(row['Protein'])
        starts   = [s.strip() for s in str(row['startPosition']).split(',')]
        ends     = [e.strip() for e in str(row['endPosition']).split(',')]
        for i, uniprot_id in enumerate(proteins):
            if uniprot_id not in sifts:
                continue
            try:
                pept_start = int(starts[i] if i < len(starts) else starts[0])
                pept_end   = int(ends[i]   if i < len(ends)   else ends[0])
            except (ValueError, TypeError, IndexError):
                continue
            pdb_positions = map_peptide_to_pdb(pept_start, pept_end, sifts[uniprot_id])
            if pdb_positions:
                result[pept_id] = pdb_positions
                break  # use first successful mapping
    return result

# Map significant peptides (used for visualization coloring)
pept_to_pdb      = map_pept_list(siglip_ribo)
pdb_sig_only     = {p: pept_to_pdb[p] for p in sig_only_ribo     if p in pept_to_pdb}
pdb_sig_corr     = {p: pept_to_pdb[p] for p in sig_and_corr_ribo if p in pept_to_pdb}

# Map ALL matched-protein peptides (needed for SASA background comparison)
all_ribo_pept_to_pdb = map_pept_list(all_ribo_pepts)

print(f'Significant peptides mapped to PDB:              {len(pept_to_pdb)}')
print(f'  Non-correlating (\u2192 {COLOR_SIG}):       {len(pdb_sig_only)}')
print(f'  Correlating     (\u2192 {COLOR_CORR}):       {len(pdb_sig_corr)}')
print(f'All matched-protein peptides mapped:             {len(all_ribo_pept_to_pdb)}')

# ------------------------------------------------------------------
# Build per-residue group sets (used for visualization and SASA)
# ------------------------------------------------------------------
def residues_from_pept_dict(pept_dict):
    """Collect all (chain, res_num) pairs covered by any peptide in the dict."""
    res_set = set()
    for positions in pept_dict.values():
        for pos in positions:
            for rn in pos['residues']:
                res_set.add((pos['chain'], rn))
    return res_set

detected_res_set = residues_from_pept_dict(all_ribo_pept_to_pdb)   # any detected peptide
sig_res_set      = residues_from_pept_dict(pept_to_pdb)             # any siglip peptide
corr_res_set     = residues_from_pept_dict(pdb_sig_corr)            # correlating peptide

# All PDB residues for matched proteins (from SIFTS) -- defines full expected coverage
sifts_residues = set()
for uniprot_id in matched_proteins:
    if uniprot_id not in sifts:
        continue
    for cm in sifts[uniprot_id]:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            sifts_residues.add((cm['chain_id'], unp_res + offset))

print(f'\nPDB residues in matched proteins (SIFTS):  {len(sifts_residues)}')
print(f'  Covered by any detected peptide:          {len(detected_res_set)}')
print(f'  Covered by significant peptide:           {len(sig_res_set)}')
print(f'  Covered by correlating peptide:           {len(corr_res_set)}')
print(f'  Not detected (no LiP-MS coverage):        {len(sifts_residues - detected_res_set)}')


Significant peptides mapped to PDB:              5
  Non-correlating (→ #ea9c8e):       2
  Correlating     (→ #d8624d):       3
All matched-protein peptides mapped:             36

PDB residues in matched proteins (SIFTS):  462
  Covered by any detected peptide:          281
  Covered by significant peptide:           50
  Covered by correlating peptide:           31
  Not detected (no LiP-MS coverage):        181


## 6. Solvent Accessibility Analysis

Computes per-residue **relative SASA (rSASA)** on the full complex using freesasa, then compares rSASA distributions across four groups (one data point = one residue, assigned to highest-priority group):

Results are cached to `data/<PDB_ID>_residue_sasa.pkl` after the first run.

In [181]:
sasa_cache_path = f'data/{struct_id}_residue_sasa.pkl'

if os.path.exists(sasa_cache_path):
    print(f'Loading cached SASA from {sasa_cache_path}...')
    with open(sasa_cache_path, 'rb') as f:
        res_sasa = pickle.load(f)
    print(f'Loaded rSASA for {len(res_sasa)} residues.')
else:
    import gemmi

    # ── 1. Parse CIF with gemmi (fast C++ library, seconds not hours) ──────────
    print(f'Parsing {PDB_ID} CIF with gemmi...')
    st    = gemmi.read_structure(cif_path)
    model = st[0]

    # ── 2. PDB format only allows 1-char chain IDs.  Remap any multi-char     ──
    #       auth_asym_ids and keep a reverse lookup so we can restore them later.
    single_used = {ch.name for ch in model if len(ch.name) == 1}
    spare       = [c for c in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789'
                   if c not in single_used]
    pdb_to_auth = {ch.name: ch.name for ch in model if len(ch.name) == 1}
    spare_idx   = 0
    for chain in model:
        if len(chain.name) > 1:
            orig          = chain.name
            new_id        = spare[spare_idx] if spare_idx < len(spare) else '?'
            spare_idx    += 1
            pdb_to_auth[new_id] = orig
            chain.name    = new_id   # rename in-place for PDB output

    print(f'  {len(list(model))} chains; {spare_idx} multi-char chain IDs remapped for PDB format')

    # ── 3. Write temporary PDB and run freesasa ────────────────────────────────
    tmp_pdb = cif_path.replace('.cif', '_freesasa_tmp.pdb')
    st.write_pdb(tmp_pdb)
    print(f'  Temporary PDB written ({os.path.getsize(tmp_pdb)/1e6:.0f} MB)')

    print('Computing SASA with freesasa...')
    try:
        fs_structure = freesasa.Structure(tmp_pdb)
        sasa_result  = freesasa.calc(fs_structure)
    finally:
        os.remove(tmp_pdb)

    # ── 4. Build per-residue rSASA lookup with original auth_asym_id labels ───
    res_sasa = {}
    for pdb_chain, chain_areas in sasa_result.residueAreas().items():
        auth_chain = pdb_to_auth.get(pdb_chain, pdb_chain)
        for res_label, res_area in chain_areas.items():
            try:
                res_num = int(str(res_label).strip())
                rsa     = res_area.relativeTotal
                if rsa is not None:
                    res_sasa[(auth_chain, res_num)] = float(rsa)
            except (ValueError, TypeError):
                pass

    with open(sasa_cache_path, 'wb') as f:
        pickle.dump(res_sasa, f)
    print(f'SASA cached to {sasa_cache_path}')

n_valid = sum(1 for v in res_sasa.values() if not np.isnan(v))
print(f'Residues with valid rSASA: {n_valid}')


Loading cached SASA from data/AF-P07284-F1_residue_sasa.pkl...
Loaded rSASA for 462 residues.
Residues with valid rSASA: 462


In [182]:
def assign_residue_group(res_key):
    if res_key in corr_res_set:
        return 'correlating'
    elif res_key in sig_res_set:
        return 'significant'
    elif res_key in detected_res_set:
        return 'detected, non-sig'
    else:
        return 'not detected'

# Build per-residue rSASA table
sasa_rows = []
for res_key in sifts_residues:
    if res_key not in res_sasa:
        continue
    rsa = res_sasa[res_key]
    if np.isnan(rsa):
        continue
    sasa_rows.append({'chain': res_key[0], 'res_num': res_key[1],
                      'lip_group': assign_residue_group(res_key), 'rSASA': rsa})

sasa_df = pd.DataFrame(sasa_rows)

group_order  = ['not detected', 'detected, non-sig', 'significant', 'correlating']
group_labels = ['Not\ndetected', 'Detected,\nnon-sig', 'Significant\n(all)', 'Correlating\nw/ metabolites']
group_colors = {
    'not detected':      COLOR_FILTERED,
    'detected, non-sig': COLOR_BASE,
    'significant':       COLOR_SIG,
    'correlating':       COLOR_CORR,
}

sasa_df['lip_group'] = pd.Categorical(sasa_df['lip_group'], categories=group_order, ordered=True)

print('Group sizes (residues):')
print(sasa_df['lip_group'].value_counts().reindex(group_order).to_string())

# ── Statistical tests (one-tailed MWU: fg > ref) ─────────────────────────────
def mwu_test(fg_vals, ref_vals):
    _, pval = stats.mannwhitneyu(fg_vals, ref_vals, alternative='greater')
    stars = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    return stars, pval

def grp_vals(grp):
    return sasa_df.loc[sasa_df['lip_group'] == grp, 'rSASA'].values

nd  = grp_vals('not detected')
dns = grp_vals('detected, non-sig')

# Set 1: all groups vs not detected; Set 2: sig/corr vs detected non-sig
test_pairs = [
    ('not detected',      'detected, non-sig', nd),
    ('not detected',      'significant',       nd),
    ('not detected',      'correlating',       nd),
    ('detected, non-sig', 'significant',       dns),
    ('detected, non-sig', 'correlating',       dns),
]
tests = []
for ref_grp, fg_grp, ref_vals in test_pairs:
    fg_vals = grp_vals(fg_grp)
    if len(fg_vals) >= 3 and len(ref_vals) >= 3:
        stars, pval = mwu_test(fg_vals, ref_vals)
        tests.append({'ref': ref_grp, 'fg': fg_grp, 'stars': stars,
                      'pval': pval})

# ── Bracket annotations ───────────────────────────────────────────────────────
y_max  = float(sasa_df['rSASA'].max())
y_step = y_max * 0.09
x_pos  = {g: i + 1 for i, g in enumerate(group_order)}
tick_h = y_step * 0.25

ref_bases = {'not detected': y_max * 1.05, 'detected, non-sig': y_max * 1.42}
ref_k     = {'not detected': 0,            'detected, non-sig': 0}

segs, txts = [], []
for t in tests:
    x1   = x_pos[t['ref']]
    x2   = x_pos[t['fg']]
    k    = ref_k[t['ref']]
    y_br = ref_bases[t['ref']] + k * y_step
    ref_k[t['ref']] += 1
    segs += [
        {'x': x1, 'xend': x2, 'y': y_br,          'yend': y_br},
        {'x': x1, 'xend': x1, 'y': y_br - tick_h, 'yend': y_br},
        {'x': x2, 'xend': x2, 'y': y_br - tick_h, 'yend': y_br},
    ]
    txts.append({
        'x':     (x1 + x2) / 2,
        'y':     y_br + y_step * 0.08,
        'label': f'{t["stars"]} p={t["pval"]:.1e}',
    })

seg_df = pd.DataFrame(segs)
txt_df = pd.DataFrame(txts)
y_lim  = ref_bases['detected, non-sig'] + ref_k['detected, non-sig'] * y_step + y_step

# ── plotnine figure ───────────────────────────────────────────────────────────
p = (
    ggplot(sasa_df, aes(x='lip_group', y='rSASA', fill='lip_group'))
    + geom_boxplot(outlier_shape='', width=0.5, show_legend=False)
    + geom_jitter(width=0.2, alpha=0.2, size=0.5,
                  color='#333333', show_legend=False)
    + geom_segment(data=seg_df,
                   mapping=aes(x='x', xend='xend', y='y', yend='yend'),
                   inherit_aes=False, size=0.5, color='black')
    + geom_text(data=txt_df,
                mapping=aes(x='x', y='y', label='label'),
                inherit_aes=False, size=6, ha='center', va='bottom')
    + scale_x_discrete(limits=group_order, labels=group_labels)
    + scale_fill_manual(values=group_colors)
    + coord_cartesian(ylim=(None, y_lim))
    + labs(x='', y='Relative SASA per residue')
    + theme_classic()
    + theme(axis_text_x=element_text(size=9))
)

os.makedirs('figure_panels', exist_ok=True)
p.save(f'figure_panels/{OUTPUT_PREFIX}_sasa_boxplot.pdf', bbox_inches='tight')
p.draw()
print(f'Saved: figure_panels/{OUTPUT_PREFIX}_sasa_boxplot.pdf')


Group sizes (residues):
lip_group
not detected         181
detected, non-sig    231
significant           19
correlating           31


/Users/xhernandez/miniconda3/envs/lipms_structure/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6.4 x 4.8 in image.
/Users/xhernandez/miniconda3/envs/lipms_structure/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figure_panels/SES1_sasa_boxplot.pdf


Saved: figure_panels/SES1_sasa_boxplot.pdf


## 7. Visualization: Peptide Level

All atoms shown as spacefill (grey). Significant peptide residues highlighted by category.

In [183]:
# Load structure string (done once, reused for both views)
print(f'Loading CIF structure ({file_size_mb:.1f} MB)...')
with open(cif_path) as f:
    cif_str = f.read()
print('Done.')

Loading CIF structure (0.4 MB)...
Done.


In [184]:
import re as _re

def _classify_chains(cif_text):
    '''Identify all RNA/DNA polymer chains in a mmCIF file.

    Covers rRNA, tRNA, mRNA, snRNA, DNA, and any other nucleic acid chain.

    Step 1: find RNA/DNA entity IDs from _entity description + _entity_poly type.
    Step 2: map entity_id -> label_asym_id via _struct_asym.
    Step 3: map label_asym_id -> auth_asym_id via _atom_site.

    Returns:
        rna_chains (set): auth_asym_ids for all RNA/DNA polymer chains.
    '''
    cut  = cif_text.find('\n_atom_site.')
    if cut < 0:
        cut = cif_text.find('\nloop_\n_atom_site.')
    meta = cif_text[:cut] if cut > 0 else cif_text[:500_000]

    # ── Step 1: entity IDs for RNA/DNA ───────────────────────────────────────
    rna_eids = set()

    # Unquoted entity type column (e.g. "1  1  polymer  tRNA")
    for m in _re.finditer(
            r'^\s*(\d+)\s+\S+\s+\S+\s+(?:tRNA|rRNA|mRNA|snRNA|snoRNA|RNA|DNA)\b',
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Single-quoted descriptions
    for m in _re.finditer(
            r"^\s*(\d+)\s+\S+\s+\S+\s+'([^']*(?:RNA|DNA|transfer|ribosomal|messenger)[^']*)'",
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Double-quoted descriptions
    for m in _re.finditer(
            r'^\s*(\d+)\s+\S+\s+\S+\s+"([^"]*(?:RNA|DNA|transfer|ribosomal|messenger)[^"]*)"',
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Also check _entity_poly.type for polyribonucleotide / polydeoxyribonucleotide
    ep_idx = meta.find('_entity_poly.')
    if ep_idx >= 0:
        blk_start = meta.rfind('loop_', 0, ep_idx)
        blk_end   = meta.find('\nloop_', ep_idx + 1)
        blk_end   = blk_end if blk_end > 0 else len(meta)
        blk       = meta[blk_start:blk_end]
        ep_hdrs   = _re.findall(r'_entity_poly\.(\S+)', blk)
        if 'entity_id' in ep_hdrs and 'type' in ep_hdrs:
            eid_col  = ep_hdrs.index('entity_id')
            type_col = ep_hdrs.index('type')
            last_hdr = f'_entity_poly.{ep_hdrs[-1]}'
            ds       = blk.find(last_hdr) + len(last_hdr)
            for line in blk[ds:].split('\n'):
                parts = line.split()
                if len(parts) > max(eid_col, type_col):
                    ptype = parts[type_col].lower()
                    if 'ribo' in ptype or 'deoxyribo' in ptype or 'dna' in ptype:
                        rna_eids.add(parts[eid_col])

    print(f'  RNA/DNA entity IDs: {rna_eids}')
    if not rna_eids:
        return set()

    # ── Step 2: label_asym_id for RNA entities ───────────────────────────────
    label_ids  = set()
    search_pos = 0
    while True:
        sa_idx = meta.find('_struct_asym.id', search_pos)
        if sa_idx < 0:
            break
        blk_start = meta.rfind('loop_', 0, sa_idx)
        blk_end   = meta.find('\nloop_', sa_idx)
        blk_end   = blk_end if blk_end > 0 else len(meta)
        blk       = meta[blk_start:blk_end]
        headers   = _re.findall(r'_struct_asym\.(\S+)', blk)
        try:
            id_col  = headers.index('id')
            eid_col = headers.index('entity_id')
        except ValueError:
            search_pos = sa_idx + 1
            continue
        last_hdr   = f'_struct_asym.{headers[-1]}'
        data_start = blk.find(last_hdr) + len(last_hdr)
        for line in blk[data_start:].split('\n'):
            parts = line.split()
            if len(parts) > max(id_col, eid_col) and parts[eid_col] in rna_eids:
                label_ids.add(parts[id_col])
        search_pos = sa_idx + 1

    print(f'  RNA/DNA label_asym_ids: {label_ids}')
    if not label_ids:
        return set()

    # ── Step 3: map label_asym_id -> auth_asym_id via _atom_site ─────────────
    atom_start = cif_text.find('\nloop_\n_atom_site.')
    if atom_start < 0:
        atom_start = cif_text.find('\n_atom_site.')
    hdr_block  = cif_text[atom_start:atom_start + 5000]
    hdr_names  = _re.findall(r'_atom_site\.(\S+)', hdr_block)
    try:
        label_col = hdr_names.index('label_asym_id')
        auth_col  = hdr_names.index('auth_asym_id')
    except ValueError:
        print('  WARNING: could not find label_asym_id/auth_asym_id columns; returning label IDs')
        return label_ids  # fallback

    last_hdr_str  = f'_atom_site.{hdr_names[-1]}'
    data_off      = cif_text.find(last_hdr_str, atom_start) + len(last_hdr_str)
    label_to_auth = {}
    remaining     = label_ids.copy()
    for line in cif_text[data_off:].split('\n'):
        if not remaining:
            break
        parts = line.split()
        if len(parts) <= max(label_col, auth_col):
            continue
        lbl = parts[label_col]
        if lbl in remaining:
            label_to_auth[lbl] = parts[auth_col]
            remaining.discard(lbl)

    auth_ids = set(label_to_auth.values())
    print(f'  RNA/DNA auth_asym_ids (for py3Dmol): {sorted(auth_ids)}')
    return auth_ids


rna_chains = _classify_chains(cif_str)
print(f'Detected RNA/DNA chains: {sorted(rna_chains) if rna_chains else "none found"}')


  RNA/DNA entity IDs: set()
Detected RNA/DNA chains: none found


In [185]:
view1 = py3Dmol.view(width=950, height=620)
view1.addModel(cif_str, 'cif')

# ── 1. Base style: semi-transparent cartoon for everything ───────────────────
# RNA/DNA chains will show as ribbons; protein atoms will be overridden below.
view1.setStyle({}, {'cartoon': {'color': '#444444', 'opacity': 0.5}})

# ── 2. Remove cartoon from protein chains (replaced by VDW surfaces below) ───
all_prot_chains = {cm['chain_id'] for uid in sifts for cm in sifts[uid]} - rna_chains
for chain in all_prot_chains:
    view1.setStyle({'chain': chain}, {})

# ── 3. Small molecules: black sticks (non-water HETATM) ──────────────────────
view1.setStyle({'hetflag': True}, {'stick': {'color': 'black', 'radius': 0.25}})
view1.setStyle({'hetflag': True, 'resn': 'HOH'}, {})   # hide water

# ── 4. All RNA/DNA chains: opaque black cartoon ───────────────────────────────
for chain in rna_chains:
    view1.setStyle({'chain': chain}, {'cartoon': {'color': 'black', 'opacity': 1.0}})

# ── 5. Group significant/correlating residues by chain ───────────────────────
corr_by_chain = defaultdict(set)
for chain, res in corr_res_set:
    corr_by_chain[chain].add(res)

sig_only_by_chain = defaultdict(set)
for chain, res in sig_res_set:
    if res not in corr_by_chain[chain]:
        sig_only_by_chain[chain].add(res)

# ── 6. Build per-chain full residue sets from SIFTS ──────────────────────────
chain_all_residues_sifts = defaultdict(set)
for uid, chain_maps in sifts.items():
    for cm in chain_maps:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            chain_all_residues_sifts[cm['chain_id']].add(unp_res + offset)

# ── 7. Classify protein chains: focus (UNIPROT_FILTER) vs background ─────────
# If UNIPROT_FILTER is set: chains for those proteins → COLOR_BASE (focus),
#   all others → COLOR_FILTERED (or hidden if COLOR_FILTERED is None).
# If UNIPROT_FILTER is None: all chains are focus → COLOR_BASE.
if UNIPROT_FILTER is not None:
    focus_chains      = {cm['chain_id'] for uid in UNIPROT_FILTER
                         for cm in sifts.get(uid, [])}
    background_chains = all_prot_chains - focus_chains
else:
    focus_chains      = all_prot_chains
    background_chains = set()

# ── 8. Render background chains ──────────────────────────────────────────────
if COLOR_FILTERED is not None:
    for chain in background_chains:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': COLOR_FILTERED},
                         {'chain': chain})

# ── 9. Render focus chain surfaces (COLOR_BASE + sig/corr highlights) ────────
for chain in focus_chains:
    colored = sig_only_by_chain[chain] | corr_by_chain[chain]

    if colored and chain in chain_all_residues_sifts:
        base_residues = sorted(chain_all_residues_sifts[chain] - colored)
        if base_residues:
            view1.addSurface(py3Dmol.VDW,
                             {'opacity': 0.8, 'color': COLOR_BASE},
                             {'chain': chain, 'resi': base_residues})
    else:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': COLOR_BASE},
                         {'chain': chain})

    # Significant non-correlating residues
    if sig_only_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_SIG},
                         {'chain': chain, 'resi': sorted(sig_only_by_chain[chain])})

    # Correlating residues
    if corr_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_CORR},
                         {'chain': chain, 'resi': sorted(corr_by_chain[chain])})

view1.zoomTo()
view1.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [186]:
os.makedirs('plots', exist_ok=True)
out_path1 = f'plots/{OUTPUT_PREFIX}_peptide_level.html'
view1.write_html(out_path1)
print(f'Saved: {out_path1}')

Saved: plots/SES1_peptide_level.html


# EF1alpha

## Parameters

Edit the cell below to re-run this analysis for a different protein complex. All downstream cells adapt automatically.

In [187]:
# ============================================================
# PARAMETERS — edit only this cell to switch to another structure
# ============================================================

# Structure source: 'pdb' downloads from RCSB, 'alphafold' downloads from AlphaFold EBI
STRUCTURE_SOURCE = 'pdb'

# RCSB PDB identifier (used when STRUCTURE_SOURCE = 'pdb')
PDB_ID           = '1F60'

# UniProt accession for AlphaFold (used when STRUCTURE_SOURCE = 'alphafold')
# e.g. 'P38179' — downloads AF-P38179-F1-model_v4.cif from alphafold.ebi.ac.uk
ALPHAFOLD_ID     = None

ANALYSIS_LABEL   = 'EF1A'   # label used in plot titles
OUTPUT_PREFIX    = 'EF1A'   # prefix for all output files

# Restrict to a subset of UniProt IDs present in both the LiP-MS data and the structure.
# None = use ALL proteins found in the intersection automatically.
# For AlphaFold, leave as None (auto) or set to [ALPHAFOLD_ID].
UNIPROT_FILTER = ["P02994"]

# Color palette (used throughout downstream cells)
COLOR_FILTERED   = None  # chains NOT in UNIPROT_FILTER (if set); use None to hide them
COLOR_BASE       = '#888888'  # focus protein chains (all proteins if UNIPROT_FILTER is None)
COLOR_SIG        = '#ea9c8e'  # significant LiP peptides, NOT metabolite-correlated
COLOR_CORR       = '#d8624d'  # significant LiP peptides, correlating with metabolites


## 1. Load LiP-MS Data

In [188]:
# Load differential structurome results from all three nutrient conditions
diff_g = pd.read_csv('results/diffStructurome_G.tsv', sep='\t', index_col='quantID')
diff_n = pd.read_csv('results/diffStructurome_N.tsv', sep='\t', index_col='quantID')
diff_p = pd.read_csv('results/diffStructurome_P.tsv', sep='\t', index_col='quantID')

# Build annotprot: union of all conditions, deduplicated by quantID
annot_cols = ['Peptide', 'Protein', 'gene_name', 'startPosition', 'endPosition', 'isTryptic']
annotprot = pd.concat([diff_g[annot_cols], diff_n[annot_cols], diff_p[annot_cols]])
annotprot = annotprot[~annotprot.index.duplicated(keep='first')]

# siglip: peptides significant (adj.P.Val < 0.05) in at least one condition
siglip_g = set(diff_g.index[(diff_g['adj.P.Val'] < 0.05) & diff_g['adj.P.Val'].notna()])
siglip_n = set(diff_n.index[(diff_n['adj.P.Val'] < 0.05) & diff_n['adj.P.Val'].notna()])
siglip_p = set(diff_p.index[(diff_p['adj.P.Val'] < 0.05) & diff_p['adj.P.Val'].notna()])
siglip = list(siglip_g | siglip_n | siglip_p)

print(f'Total peptides in annotprot: {len(annotprot)}')
print(f'Significant peptides (siglip): {len(siglip)}')
print(f'  G: {len(siglip_g)}, N: {len(siglip_n)}, P: {len(siglip_p)}')
annotprot.head(3)

Total peptides in annotprot: 29732
Significant peptides (siglip): 5540
  G: 3204, N: 1112, P: 1782


,Peptide,Protein,gene_name,startPosition,endPosition,isTryptic
quantID,,,,,,
EMIISSLSPLEPHSQNEILQYYL,EMIISSLSPLEPHSQNEILQYYL,P12945,NAT1,832,854,Specific
SFDDLHK,SFDDLHK,P17709,GLK1,2,8,Specific-N
EQNEDLGLSDIKVE,EQNEDLGLSDIKVE,P53303,ZPR1,473,486,Specific


## 2. Load Correlation Data

In [189]:
# Load LiP-MS vs Metabolomics correlation results
# X1 = peptide (quantID), X2 = metabolite, lfdr = local false discovery rate
cors = pd.read_csv('source_data/LiPMS_vs_Metab.tsv', sep='\t', index_col=0)
print(f'Columns: {list(cors.columns)}')
print(f'Total peptide-metabolite pairs tested: {len(cors)}')

# Correlating peptides: those with lfdr < 0.05 in at least one metabolite pair
correlating_pepts = set(cors.loc[cors['lfdr'] < 0.05, 'X1'])
print(f'Peptides with significant metabolite correlations (lfdr < 0.05): {len(correlating_pepts)}')

# Classify siglip peptides into two categories:
#   sig_only     → significant LiP but NOT metabolite-correlated → color #4c7b6a
#   sig_and_corr → significant LiP AND metabolite-correlated   → color #d8624d
sig_only     = [p for p in siglip if p not in correlating_pepts]
sig_and_corr = [p for p in siglip if p in correlating_pepts]

print(f'\nSignificant, non-correlating (→ #4c7b6a): {len(sig_only)}')
print(f'Significant AND correlating  (→ #d8624d): {len(sig_and_corr)}')

Columns: ['X1', 'X2', 'pval', 'qval', 'lfdr', 'R2', 'annot_pept', 'annot_metab', 'GO_translation']
Total peptide-metabolite pairs tested: 1429866
Peptides with significant metabolite correlations (lfdr < 0.05): 1558

Significant, non-correlating (→ #4c7b6a): 3982
Significant AND correlating  (→ #d8624d): 1558


## 3. Fetch Structure and SIFTS Mapping


In [190]:
os.makedirs('data', exist_ok=True)

if STRUCTURE_SOURCE == 'alphafold':
    struct_id = f'AF-{ALPHAFOLD_ID}-F1'
    cif_path  = f'data/{struct_id}.cif'
    if not os.path.exists(cif_path):
        print(f'Downloading AlphaFold structure for {ALPHAFOLD_ID}...')
        url  = f'https://alphafold.ebi.ac.uk/files/{struct_id}-model_v6.cif'
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cif_path, 'w') as f:
            f.write(resp.text)
        print(f'Saved to {cif_path}')
    else:
        print(f'Using cached {cif_path}')
else:
    struct_id = PDB_ID
    cif_path  = f'data/{PDB_ID}.cif'
    if not os.path.exists(cif_path):
        print(f'Downloading {PDB_ID} from RCSB (may take a minute for large structures)...')
        url  = f'https://files.rcsb.org/download/{PDB_ID}.cif'
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cif_path, 'w') as f:
            f.write(resp.text)
        print(f'Saved to {cif_path}')
    else:
        print(f'Using cached {cif_path}')

file_size_mb = os.path.getsize(cif_path) / 1e6
print(f'File size: {file_size_mb:.1f} MB')


Using cached data/1F60.cif
File size: 0.5 MB


In [191]:
def get_sifts_mapping(pdb_id):
    """
    Fetch UniProt -> PDB chain/residue mapping from PDBe SIFTS API.
    Returns:
        sifts:      {uniprot_id: [{'chain_id', 'unp_start', 'unp_end', 'pdb_start', 'pdb_end'}]}
        sifts_ident: {uniprot_id: identifier_string}  e.g. 'RL2A_YEAST'
    """
    url = f'https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id.lower()}'
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    pdb_key = pdb_id.lower()

    sifts       = {}
    sifts_ident = {}
    for uniprot_id, info in data[pdb_key]['UniProt'].items():
        chain_maps = []
        for m in info['mappings']:
            chain_maps.append({
                'chain_id': m['chain_id'],
                'unp_start': m['unp_start'],
                'unp_end':   m['unp_end'],
                'pdb_start': m['start']['residue_number'],
                'pdb_end':   m['end']['residue_number'],
            })
        sifts[uniprot_id]       = chain_maps
        sifts_ident[uniprot_id] = info.get('identifier', '')
    return sifts, sifts_ident


if STRUCTURE_SOURCE == 'alphafold':
    # AlphaFold models always use chain 'A' with residue numbers = UniProt positions.
    # Fetch exact sequence length from UniProt REST API.
    print(f'AlphaFold mode: fetching sequence length for {ALPHAFOLD_ID} from UniProt...')
    try:
        _resp = requests.get(
            f'https://rest.uniprot.org/uniprotkb/{ALPHAFOLD_ID}.json', timeout=30)
        _resp.raise_for_status()
        _seq_len = _resp.json()['sequence']['length']
        print(f'  Sequence length: {_seq_len} aa')
    except Exception as e:
        print(f'  UniProt API failed ({e}), falling back to length=10000')
        _seq_len = 10000
    sifts = {
        ALPHAFOLD_ID: [{'chain_id': 'A',
                         'unp_start': 1,    'unp_end': _seq_len,
                         'pdb_start': 1,    'pdb_end': _seq_len}]
    }
    sifts_ident = {ALPHAFOLD_ID: ALPHAFOLD_ID}
    print(f'Synthetic SIFTS: chain A, residues 1-{_seq_len} -> {ALPHAFOLD_ID}')
else:
    print(f'Fetching SIFTS mapping for {PDB_ID}...')
    sifts, sifts_ident = get_sifts_mapping(PDB_ID)
    print(f'Proteins with SIFTS mappings in {PDB_ID}: {len(sifts)}')
    print(f'Example UniProt IDs: {list(sifts.keys())[:8]}')


Fetching SIFTS mapping for 1F60...
Proteins with SIFTS mappings in 1F60: 2
Example UniProt IDs: ['P02994', 'P32471']


## 4. Identify Peptides

In [192]:
# Find which LiP-MS proteins overlap with proteins in the ribosome structure
# Handle comma-separated protein IDs (ambiguously assigned peptides)
def _proteins_of(protein_str):
    """Return list of individual UniProt IDs from a possibly comma-separated string."""
    return [p.strip() for p in str(protein_str).split(',')]

lipm_proteins     = {p for prot_str in annotprot['Protein'] for p in _proteins_of(prot_str)}
ribosome_proteins = set(sifts.keys())
matched_proteins  = lipm_proteins & ribosome_proteins

# Apply optional UniProt filter (set UNIPROT_FILTER in the Parameters cell)
if UNIPROT_FILTER is not None:
    matched_proteins = matched_proteins & set(UNIPROT_FILTER)
    print(f'Restricted to {len(UNIPROT_FILTER)} specified UniProt IDs \u2192 {len(matched_proteins)} matched')

print(f'Proteins in LiP-MS dataset:  {len(lipm_proteins)}')
print(f'Proteins in {PDB_ID}:         {len(ribosome_proteins)}')
print(f'Matched proteins:            {len(matched_proteins)}')

# Filter peptide lists to matched proteins only
def filter_to_target(pept_list):
    return [p for p in pept_list
            if p in annotprot.index and
            any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]

siglip_ribo       = filter_to_target(siglip)
sig_only_ribo     = filter_to_target(sig_only)
sig_and_corr_ribo = filter_to_target(sig_and_corr)

print(f'\nPeptides from matched proteins in siglip:      {len(siglip_ribo)}')
print(f'  Sig, non-correlating (\u2192 {COLOR_SIG}):  {len(sig_only_ribo)}')
print(f'  Sig AND correlating  (\u2192 {COLOR_CORR}):  {len(sig_and_corr_ribo)}')

# All peptides from matched proteins (background for SASA analysis)
all_ribo_pepts = [p for p in annotprot.index
                  if any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]
print(f'\nAll matched-protein peptides (SASA background): {len(all_ribo_pepts)}')

# Show matched proteins and gene names
ribo_annot = (annotprot[annotprot['Protein'].apply(
                  lambda x: any(p in matched_proteins for p in _proteins_of(x)))]
              [['Protein', 'gene_name']].drop_duplicates().sort_values('gene_name'))
print(f'\nMatched proteins ({len(ribo_annot)}):')
display(ribo_annot)


Restricted to 1 specified UniProt IDs → 1 matched
Proteins in LiP-MS dataset:  2510
Proteins in 1F60:         2
Matched proteins:            1

Peptides from matched proteins in siglip:      31
  Sig, non-correlating (→ #ea9c8e):  21
  Sig AND correlating  (→ #d8624d):  10

All matched-protein peptides (SASA background): 126

Matched proteins (1):


,Protein,gene_name
quantID,,
LEQGVPGDNVGFNVK,P02994,TEF1;TEF2


## 5. Map Peptide Positions to PDB Residues

Uses SIFTS to convert UniProt sequence positions (startPosition, endPosition from LiP-MS) to PDB chain + residue numbers.

In [193]:
from collections import defaultdict

def map_peptide_to_pdb(pept_start, pept_end, chain_maps):
    """
    Map UniProt peptide [pept_start, pept_end] to PDB chain + residue list.
    Handles cases where a peptide spans multiple mapping segments.
    Returns: [{'chain': chain_id, 'residues': [res1, res2, ...]}]
    """
    results = []
    for cm in chain_maps:
        ov_start = max(pept_start, cm['unp_start'])
        ov_end   = min(pept_end,   cm['unp_end'])
        if ov_start <= ov_end:
            offset  = cm['pdb_start'] - cm['unp_start']
            pdb_res = list(range(ov_start + offset, ov_end + offset + 1))
            results.append({'chain': cm['chain_id'], 'residues': pdb_res})
    return results

def map_pept_list(pept_list):
    """Map a list of peptide IDs to PDB chain + residue positions.
    Handles peptides ambiguously assigned to multiple comma-separated proteins:
    tries each protein with its corresponding start/end position in order,
    and uses the first one that maps successfully to the PDB structure."""
    result = {}
    for pept_id in pept_list:
        if pept_id not in annotprot.index:
            continue
        row      = annotprot.loc[pept_id]
        proteins = _proteins_of(row['Protein'])
        starts   = [s.strip() for s in str(row['startPosition']).split(',')]
        ends     = [e.strip() for e in str(row['endPosition']).split(',')]
        for i, uniprot_id in enumerate(proteins):
            if uniprot_id not in sifts:
                continue
            try:
                pept_start = int(starts[i] if i < len(starts) else starts[0])
                pept_end   = int(ends[i]   if i < len(ends)   else ends[0])
            except (ValueError, TypeError, IndexError):
                continue
            pdb_positions = map_peptide_to_pdb(pept_start, pept_end, sifts[uniprot_id])
            if pdb_positions:
                result[pept_id] = pdb_positions
                break  # use first successful mapping
    return result

# Map significant peptides (used for visualization coloring)
pept_to_pdb      = map_pept_list(siglip_ribo)
pdb_sig_only     = {p: pept_to_pdb[p] for p in sig_only_ribo     if p in pept_to_pdb}
pdb_sig_corr     = {p: pept_to_pdb[p] for p in sig_and_corr_ribo if p in pept_to_pdb}

# Map ALL matched-protein peptides (needed for SASA background comparison)
all_ribo_pept_to_pdb = map_pept_list(all_ribo_pepts)

print(f'Significant peptides mapped to PDB:              {len(pept_to_pdb)}')
print(f'  Non-correlating (\u2192 {COLOR_SIG}):       {len(pdb_sig_only)}')
print(f'  Correlating     (\u2192 {COLOR_CORR}):       {len(pdb_sig_corr)}')
print(f'All matched-protein peptides mapped:             {len(all_ribo_pept_to_pdb)}')

# ------------------------------------------------------------------
# Build per-residue group sets (used for visualization and SASA)
# ------------------------------------------------------------------
def residues_from_pept_dict(pept_dict):
    """Collect all (chain, res_num) pairs covered by any peptide in the dict."""
    res_set = set()
    for positions in pept_dict.values():
        for pos in positions:
            for rn in pos['residues']:
                res_set.add((pos['chain'], rn))
    return res_set

detected_res_set = residues_from_pept_dict(all_ribo_pept_to_pdb)   # any detected peptide
sig_res_set      = residues_from_pept_dict(pept_to_pdb)             # any siglip peptide
corr_res_set     = residues_from_pept_dict(pdb_sig_corr)            # correlating peptide

# All PDB residues for matched proteins (from SIFTS) -- defines full expected coverage
sifts_residues = set()
for uniprot_id in matched_proteins:
    if uniprot_id not in sifts:
        continue
    for cm in sifts[uniprot_id]:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            sifts_residues.add((cm['chain_id'], unp_res + offset))

print(f'\nPDB residues in matched proteins (SIFTS):  {len(sifts_residues)}')
print(f'  Covered by any detected peptide:          {len(detected_res_set)}')
print(f'  Covered by significant peptide:           {len(sig_res_set)}')
print(f'  Covered by correlating peptide:           {len(corr_res_set)}')
print(f'  Not detected (no LiP-MS coverage):        {len(sifts_residues - detected_res_set)}')


Significant peptides mapped to PDB:              31
  Non-correlating (→ #ea9c8e):       21
  Correlating     (→ #d8624d):       10
All matched-protein peptides mapped:             126

PDB residues in matched proteins (SIFTS):  458
  Covered by any detected peptide:          295
  Covered by significant peptide:           177
  Covered by correlating peptide:           100
  Not detected (no LiP-MS coverage):        163


## 7. Visualization: Peptide Level

All atoms shown as spacefill (grey). Significant peptide residues highlighted by category.

In [194]:
# Load structure string (done once, reused for both views)
print(f'Loading CIF structure ({file_size_mb:.1f} MB)...')
with open(cif_path) as f:
    cif_str = f.read()
print('Done.')

Loading CIF structure (0.5 MB)...
Done.


In [195]:
import re as _re

def _classify_chains(cif_text):
    '''Identify all RNA/DNA polymer chains in a mmCIF file.

    Covers rRNA, tRNA, mRNA, snRNA, DNA, and any other nucleic acid chain.

    Step 1: find RNA/DNA entity IDs from _entity description + _entity_poly type.
    Step 2: map entity_id -> label_asym_id via _struct_asym.
    Step 3: map label_asym_id -> auth_asym_id via _atom_site.

    Returns:
        rna_chains (set): auth_asym_ids for all RNA/DNA polymer chains.
    '''
    cut  = cif_text.find('\n_atom_site.')
    if cut < 0:
        cut = cif_text.find('\nloop_\n_atom_site.')
    meta = cif_text[:cut] if cut > 0 else cif_text[:500_000]

    # ── Step 1: entity IDs for RNA/DNA ───────────────────────────────────────
    rna_eids = set()

    # Unquoted entity type column (e.g. "1  1  polymer  tRNA")
    for m in _re.finditer(
            r'^\s*(\d+)\s+\S+\s+\S+\s+(?:tRNA|rRNA|mRNA|snRNA|snoRNA|RNA|DNA)\b',
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Single-quoted descriptions
    for m in _re.finditer(
            r"^\s*(\d+)\s+\S+\s+\S+\s+'([^']*(?:RNA|DNA|transfer|ribosomal|messenger)[^']*)'",
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Double-quoted descriptions
    for m in _re.finditer(
            r'^\s*(\d+)\s+\S+\s+\S+\s+"([^"]*(?:RNA|DNA|transfer|ribosomal|messenger)[^"]*)"',
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Also check _entity_poly.type for polyribonucleotide / polydeoxyribonucleotide
    ep_idx = meta.find('_entity_poly.')
    if ep_idx >= 0:
        blk_start = meta.rfind('loop_', 0, ep_idx)
        blk_end   = meta.find('\nloop_', ep_idx + 1)
        blk_end   = blk_end if blk_end > 0 else len(meta)
        blk       = meta[blk_start:blk_end]
        ep_hdrs   = _re.findall(r'_entity_poly\.(\S+)', blk)
        if 'entity_id' in ep_hdrs and 'type' in ep_hdrs:
            eid_col  = ep_hdrs.index('entity_id')
            type_col = ep_hdrs.index('type')
            last_hdr = f'_entity_poly.{ep_hdrs[-1]}'
            ds       = blk.find(last_hdr) + len(last_hdr)
            for line in blk[ds:].split('\n'):
                parts = line.split()
                if len(parts) > max(eid_col, type_col):
                    ptype = parts[type_col].lower()
                    if 'ribo' in ptype or 'deoxyribo' in ptype or 'dna' in ptype:
                        rna_eids.add(parts[eid_col])

    print(f'  RNA/DNA entity IDs: {rna_eids}')
    if not rna_eids:
        return set()

    # ── Step 2: label_asym_id for RNA entities ───────────────────────────────
    label_ids  = set()
    search_pos = 0
    while True:
        sa_idx = meta.find('_struct_asym.id', search_pos)
        if sa_idx < 0:
            break
        blk_start = meta.rfind('loop_', 0, sa_idx)
        blk_end   = meta.find('\nloop_', sa_idx)
        blk_end   = blk_end if blk_end > 0 else len(meta)
        blk       = meta[blk_start:blk_end]
        headers   = _re.findall(r'_struct_asym\.(\S+)', blk)
        try:
            id_col  = headers.index('id')
            eid_col = headers.index('entity_id')
        except ValueError:
            search_pos = sa_idx + 1
            continue
        last_hdr   = f'_struct_asym.{headers[-1]}'
        data_start = blk.find(last_hdr) + len(last_hdr)
        for line in blk[data_start:].split('\n'):
            parts = line.split()
            if len(parts) > max(id_col, eid_col) and parts[eid_col] in rna_eids:
                label_ids.add(parts[id_col])
        search_pos = sa_idx + 1

    print(f'  RNA/DNA label_asym_ids: {label_ids}')
    if not label_ids:
        return set()

    # ── Step 3: map label_asym_id -> auth_asym_id via _atom_site ─────────────
    atom_start = cif_text.find('\nloop_\n_atom_site.')
    if atom_start < 0:
        atom_start = cif_text.find('\n_atom_site.')
    hdr_block  = cif_text[atom_start:atom_start + 5000]
    hdr_names  = _re.findall(r'_atom_site\.(\S+)', hdr_block)
    try:
        label_col = hdr_names.index('label_asym_id')
        auth_col  = hdr_names.index('auth_asym_id')
    except ValueError:
        print('  WARNING: could not find label_asym_id/auth_asym_id columns; returning label IDs')
        return label_ids  # fallback

    last_hdr_str  = f'_atom_site.{hdr_names[-1]}'
    data_off      = cif_text.find(last_hdr_str, atom_start) + len(last_hdr_str)
    label_to_auth = {}
    remaining     = label_ids.copy()
    for line in cif_text[data_off:].split('\n'):
        if not remaining:
            break
        parts = line.split()
        if len(parts) <= max(label_col, auth_col):
            continue
        lbl = parts[label_col]
        if lbl in remaining:
            label_to_auth[lbl] = parts[auth_col]
            remaining.discard(lbl)

    auth_ids = set(label_to_auth.values())
    print(f'  RNA/DNA auth_asym_ids (for py3Dmol): {sorted(auth_ids)}')
    return auth_ids


rna_chains = _classify_chains(cif_str)
print(f'Detected RNA/DNA chains: {sorted(rna_chains) if rna_chains else "none found"}')


  RNA/DNA entity IDs: set()
Detected RNA/DNA chains: none found


In [196]:
view1 = py3Dmol.view(width=950, height=620)
view1.addModel(cif_str, 'cif')

# ── 1. Base style: semi-transparent cartoon for everything ───────────────────
# RNA/DNA chains will show as ribbons; protein atoms will be overridden below.
view1.setStyle({}, {'cartoon': {'color': '#444444', 'opacity': 0.5}})

# ── 2. Remove cartoon from protein chains (replaced by VDW surfaces below) ───
all_prot_chains = {cm['chain_id'] for uid in sifts for cm in sifts[uid]} - rna_chains
for chain in all_prot_chains:
    view1.setStyle({'chain': chain}, {})

# ── 3. Small molecules: black sticks (non-water HETATM) ──────────────────────
view1.setStyle({'hetflag': True}, {'stick': {'color': 'black', 'radius': 0.25}})
view1.setStyle({'hetflag': True, 'resn': 'HOH'}, {})   # hide water

# ── 4. All RNA/DNA chains: opaque black cartoon ───────────────────────────────
for chain in rna_chains:
    view1.setStyle({'chain': chain}, {'cartoon': {'color': 'black', 'opacity': 1.0}})

# ── 5. Group significant/correlating residues by chain ───────────────────────
corr_by_chain = defaultdict(set)
for chain, res in corr_res_set:
    corr_by_chain[chain].add(res)

sig_only_by_chain = defaultdict(set)
for chain, res in sig_res_set:
    if res not in corr_by_chain[chain]:
        sig_only_by_chain[chain].add(res)

# ── 6. Build per-chain full residue sets from SIFTS ──────────────────────────
chain_all_residues_sifts = defaultdict(set)
for uid, chain_maps in sifts.items():
    for cm in chain_maps:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            chain_all_residues_sifts[cm['chain_id']].add(unp_res + offset)

# ── 7. Classify protein chains: focus (UNIPROT_FILTER) vs background ─────────
# If UNIPROT_FILTER is set: chains for those proteins → COLOR_BASE (focus),
#   all others → COLOR_FILTERED (or hidden if COLOR_FILTERED is None).
# If UNIPROT_FILTER is None: all chains are focus → COLOR_BASE.
if UNIPROT_FILTER is not None:
    focus_chains      = {cm['chain_id'] for uid in UNIPROT_FILTER
                         for cm in sifts.get(uid, [])}
    background_chains = all_prot_chains - focus_chains
else:
    focus_chains      = all_prot_chains
    background_chains = set()

# ── 8. Render background chains ──────────────────────────────────────────────
if COLOR_FILTERED is not None:
    for chain in background_chains:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': COLOR_FILTERED},
                         {'chain': chain})

# ── 9. Render focus chain surfaces (COLOR_BASE + sig/corr highlights) ────────
for chain in focus_chains:
    colored = sig_only_by_chain[chain] | corr_by_chain[chain]

    if colored and chain in chain_all_residues_sifts:
        base_residues = sorted(chain_all_residues_sifts[chain] - colored)
        if base_residues:
            view1.addSurface(py3Dmol.VDW,
                             {'opacity': 0.8, 'color': COLOR_BASE},
                             {'chain': chain, 'resi': base_residues})
    else:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': COLOR_BASE},
                         {'chain': chain})

    # Significant non-correlating residues
    if sig_only_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_SIG},
                         {'chain': chain, 'resi': sorted(sig_only_by_chain[chain])})

    # Correlating residues
    if corr_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_CORR},
                         {'chain': chain, 'resi': sorted(corr_by_chain[chain])})

view1.zoomTo()
view1.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [197]:
os.makedirs('plots', exist_ok=True)
out_path1 = f'plots/{OUTPUT_PREFIX}_peptide_level.html'
view1.write_html(out_path1)
print(f'Saved: {out_path1}')

Saved: plots/EF1A_peptide_level.html


# MES1

## Parameters

Edit the cell below to re-run this analysis for a different protein complex. All downstream cells adapt automatically.

In [233]:
# ============================================================
# PARAMETERS — edit only this cell to switch to another structure
# ============================================================

# Structure source: 'pdb' downloads from RCSB, 'alphafold' downloads from AlphaFold EBI
STRUCTURE_SOURCE = 'alphafold'

# RCSB PDB identifier (used when STRUCTURE_SOURCE = 'pdb')
PDB_ID           = None

# UniProt accession for AlphaFold (used when STRUCTURE_SOURCE = 'alphafold')
# e.g. 'P38179' — downloads AF-P38179-F1-model_v4.cif from alphafold.ebi.ac.uk
ALPHAFOLD_ID     = 'P00958'

ANALYSIS_LABEL   = 'MES1'   # label used in plot titles
OUTPUT_PREFIX    = 'MES1'   # prefix for all output files

# Restrict to a subset of UniProt IDs present in both the LiP-MS data and the structure.
# None = use ALL proteins found in the intersection automatically.
# For AlphaFold, leave as None (auto) or set to [ALPHAFOLD_ID].
UNIPROT_FILTER = None

# Color palette (used throughout downstream cells)
COLOR_FILTERED   = None  # chains NOT in UNIPROT_FILTER (if set); use None to hide them
COLOR_BASE       = '#888888'  # focus protein chains (all proteins if UNIPROT_FILTER is None)
COLOR_SIG        = '#ea9c8e'  # significant LiP peptides, NOT metabolite-correlated
COLOR_CORR       = '#d8624d'  # significant LiP peptides, correlating with metabolites


## 1. Load LiP-MS Data

In [234]:
# Load differential structurome results from all three nutrient conditions
diff_g = pd.read_csv('results/diffStructurome_G.tsv', sep='\t', index_col='quantID')
diff_n = pd.read_csv('results/diffStructurome_N.tsv', sep='\t', index_col='quantID')
diff_p = pd.read_csv('results/diffStructurome_P.tsv', sep='\t', index_col='quantID')

# Build annotprot: union of all conditions, deduplicated by quantID
annot_cols = ['Peptide', 'Protein', 'gene_name', 'startPosition', 'endPosition', 'isTryptic']
annotprot = pd.concat([diff_g[annot_cols], diff_n[annot_cols], diff_p[annot_cols]])
annotprot = annotprot[~annotprot.index.duplicated(keep='first')]

# siglip: peptides significant (adj.P.Val < 0.05) in at least one condition
siglip_g = set(diff_g.index[(diff_g['adj.P.Val'] < 0.05) & diff_g['adj.P.Val'].notna()])
siglip_n = set(diff_n.index[(diff_n['adj.P.Val'] < 0.05) & diff_n['adj.P.Val'].notna()])
siglip_p = set(diff_p.index[(diff_p['adj.P.Val'] < 0.05) & diff_p['adj.P.Val'].notna()])
siglip = list(siglip_g | siglip_n | siglip_p)

print(f'Total peptides in annotprot: {len(annotprot)}')
print(f'Significant peptides (siglip): {len(siglip)}')
print(f'  G: {len(siglip_g)}, N: {len(siglip_n)}, P: {len(siglip_p)}')
annotprot.head(3)

Total peptides in annotprot: 29732
Significant peptides (siglip): 5540
  G: 3204, N: 1112, P: 1782


,Peptide,Protein,gene_name,startPosition,endPosition,isTryptic
quantID,,,,,,
EMIISSLSPLEPHSQNEILQYYL,EMIISSLSPLEPHSQNEILQYYL,P12945,NAT1,832,854,Specific
SFDDLHK,SFDDLHK,P17709,GLK1,2,8,Specific-N
EQNEDLGLSDIKVE,EQNEDLGLSDIKVE,P53303,ZPR1,473,486,Specific


## 2. Load Correlation Data

In [235]:
# Load LiP-MS vs Metabolomics correlation results
# X1 = peptide (quantID), X2 = metabolite, lfdr = local false discovery rate
cors = pd.read_csv('source_data/LiPMS_vs_Metab.tsv', sep='\t', index_col=0)
print(f'Columns: {list(cors.columns)}')
print(f'Total peptide-metabolite pairs tested: {len(cors)}')

# Correlating peptides: those with lfdr < 0.05 in at least one metabolite pair
correlating_pepts = set(cors.loc[cors['lfdr'] < 0.05, 'X1'])
print(f'Peptides with significant metabolite correlations (lfdr < 0.05): {len(correlating_pepts)}')

# Classify siglip peptides into two categories:
#   sig_only     → significant LiP but NOT metabolite-correlated → color #4c7b6a
#   sig_and_corr → significant LiP AND metabolite-correlated   → color #d8624d
sig_only     = [p for p in siglip if p not in correlating_pepts]
sig_and_corr = [p for p in siglip if p in correlating_pepts]

print(f'\nSignificant, non-correlating (→ #4c7b6a): {len(sig_only)}')
print(f'Significant AND correlating  (→ #d8624d): {len(sig_and_corr)}')

Columns: ['X1', 'X2', 'pval', 'qval', 'lfdr', 'R2', 'annot_pept', 'annot_metab', 'GO_translation']
Total peptide-metabolite pairs tested: 1429866
Peptides with significant metabolite correlations (lfdr < 0.05): 1558

Significant, non-correlating (→ #4c7b6a): 3982
Significant AND correlating  (→ #d8624d): 1558


## 3. Fetch Structure and SIFTS Mapping


In [236]:
os.makedirs('data', exist_ok=True)

if STRUCTURE_SOURCE == 'alphafold':
    struct_id = f'AF-{ALPHAFOLD_ID}-F1'
    cif_path  = f'data/{struct_id}.cif'
    if not os.path.exists(cif_path):
        print(f'Downloading AlphaFold structure for {ALPHAFOLD_ID}...')
        url  = f'https://alphafold.ebi.ac.uk/files/{struct_id}-model_v6.cif'
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cif_path, 'w') as f:
            f.write(resp.text)
        print(f'Saved to {cif_path}')
    else:
        print(f'Using cached {cif_path}')
else:
    struct_id = PDB_ID
    cif_path  = f'data/{PDB_ID}.cif'
    if not os.path.exists(cif_path):
        print(f'Downloading {PDB_ID} from RCSB (may take a minute for large structures)...')
        url  = f'https://files.rcsb.org/download/{PDB_ID}.cif'
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cif_path, 'w') as f:
            f.write(resp.text)
        print(f'Saved to {cif_path}')
    else:
        print(f'Using cached {cif_path}')

file_size_mb = os.path.getsize(cif_path) / 1e6
print(f'File size: {file_size_mb:.1f} MB')


Using cached data/AF-P00958-F1.cif
File size: 0.7 MB


In [237]:
def get_sifts_mapping(pdb_id):
    """
    Fetch UniProt -> PDB chain/residue mapping from PDBe SIFTS API.
    Returns:
        sifts:      {uniprot_id: [{'chain_id', 'unp_start', 'unp_end', 'pdb_start', 'pdb_end'}]}
        sifts_ident: {uniprot_id: identifier_string}  e.g. 'RL2A_YEAST'
    """
    url = f'https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id.lower()}'
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    pdb_key = pdb_id.lower()

    sifts       = {}
    sifts_ident = {}
    for uniprot_id, info in data[pdb_key]['UniProt'].items():
        chain_maps = []
        for m in info['mappings']:
            chain_maps.append({
                'chain_id': m['chain_id'],
                'unp_start': m['unp_start'],
                'unp_end':   m['unp_end'],
                'pdb_start': m['start']['residue_number'],
                'pdb_end':   m['end']['residue_number'],
            })
        sifts[uniprot_id]       = chain_maps
        sifts_ident[uniprot_id] = info.get('identifier', '')
    return sifts, sifts_ident


if STRUCTURE_SOURCE == 'alphafold':
    # AlphaFold models always use chain 'A' with residue numbers = UniProt positions.
    # Fetch exact sequence length from UniProt REST API.
    print(f'AlphaFold mode: fetching sequence length for {ALPHAFOLD_ID} from UniProt...')
    try:
        _resp = requests.get(
            f'https://rest.uniprot.org/uniprotkb/{ALPHAFOLD_ID}.json', timeout=30)
        _resp.raise_for_status()
        _seq_len = _resp.json()['sequence']['length']
        print(f'  Sequence length: {_seq_len} aa')
    except Exception as e:
        print(f'  UniProt API failed ({e}), falling back to length=10000')
        _seq_len = 10000
    sifts = {
        ALPHAFOLD_ID: [{'chain_id': 'A',
                         'unp_start': 1,    'unp_end': _seq_len,
                         'pdb_start': 1,    'pdb_end': _seq_len}]
    }
    sifts_ident = {ALPHAFOLD_ID: ALPHAFOLD_ID}
    print(f'Synthetic SIFTS: chain A, residues 1-{_seq_len} -> {ALPHAFOLD_ID}')
else:
    print(f'Fetching SIFTS mapping for {PDB_ID}...')
    sifts, sifts_ident = get_sifts_mapping(PDB_ID)
    print(f'Proteins with SIFTS mappings in {PDB_ID}: {len(sifts)}')
    print(f'Example UniProt IDs: {list(sifts.keys())[:8]}')


AlphaFold mode: fetching sequence length for P00958 from UniProt...
  Sequence length: 751 aa
Synthetic SIFTS: chain A, residues 1-751 -> P00958


## 4. Identify Peptides

In [238]:
# Find which LiP-MS proteins overlap with proteins in the ribosome structure
# Handle comma-separated protein IDs (ambiguously assigned peptides)
def _proteins_of(protein_str):
    """Return list of individual UniProt IDs from a possibly comma-separated string."""
    return [p.strip() for p in str(protein_str).split(',')]

lipm_proteins     = {p for prot_str in annotprot['Protein'] for p in _proteins_of(prot_str)}
ribosome_proteins = set(sifts.keys())
matched_proteins  = lipm_proteins & ribosome_proteins

# Apply optional UniProt filter (set UNIPROT_FILTER in the Parameters cell)
if UNIPROT_FILTER is not None:
    matched_proteins = matched_proteins & set(UNIPROT_FILTER)
    print(f'Restricted to {len(UNIPROT_FILTER)} specified UniProt IDs \u2192 {len(matched_proteins)} matched')

print(f'Proteins in LiP-MS dataset:  {len(lipm_proteins)}')
print(f'Proteins in {PDB_ID}:         {len(ribosome_proteins)}')
print(f'Matched proteins:            {len(matched_proteins)}')

# Filter peptide lists to matched proteins only
def filter_to_target(pept_list):
    return [p for p in pept_list
            if p in annotprot.index and
            any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]

siglip_ribo       = filter_to_target(siglip)
sig_only_ribo     = filter_to_target(sig_only)
sig_and_corr_ribo = filter_to_target(sig_and_corr)

print(f'\nPeptides from matched proteins in siglip:      {len(siglip_ribo)}')
print(f'  Sig, non-correlating (\u2192 {COLOR_SIG}):  {len(sig_only_ribo)}')
print(f'  Sig AND correlating  (\u2192 {COLOR_CORR}):  {len(sig_and_corr_ribo)}')

# All peptides from matched proteins (background for SASA analysis)
all_ribo_pepts = [p for p in annotprot.index
                  if any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]
print(f'\nAll matched-protein peptides (SASA background): {len(all_ribo_pepts)}')

# Show matched proteins and gene names
ribo_annot = (annotprot[annotprot['Protein'].apply(
                  lambda x: any(p in matched_proteins for p in _proteins_of(x)))]
              [['Protein', 'gene_name']].drop_duplicates().sort_values('gene_name'))
print(f'\nMatched proteins ({len(ribo_annot)}):')
display(ribo_annot)


Proteins in LiP-MS dataset:  2510
Proteins in None:         1
Matched proteins:            1

Peptides from matched proteins in siglip:      13
  Sig, non-correlating (→ #ea9c8e):  11
  Sig AND correlating  (→ #d8624d):  2

All matched-protein peptides (SASA background): 46

Matched proteins (1):


,Protein,gene_name
quantID,,
NTKEPFLLFDANAILR,P00958,MES1


## 5. Map Peptide Positions to PDB Residues

Uses SIFTS to convert UniProt sequence positions (startPosition, endPosition from LiP-MS) to PDB chain + residue numbers.

In [239]:
from collections import defaultdict

def map_peptide_to_pdb(pept_start, pept_end, chain_maps):
    """
    Map UniProt peptide [pept_start, pept_end] to PDB chain + residue list.
    Handles cases where a peptide spans multiple mapping segments.
    Returns: [{'chain': chain_id, 'residues': [res1, res2, ...]}]
    """
    results = []
    for cm in chain_maps:
        ov_start = max(pept_start, cm['unp_start'])
        ov_end   = min(pept_end,   cm['unp_end'])
        if ov_start <= ov_end:
            offset  = cm['pdb_start'] - cm['unp_start']
            pdb_res = list(range(ov_start + offset, ov_end + offset + 1))
            results.append({'chain': cm['chain_id'], 'residues': pdb_res})
    return results

def map_pept_list(pept_list):
    """Map a list of peptide IDs to PDB chain + residue positions.
    Handles peptides ambiguously assigned to multiple comma-separated proteins:
    tries each protein with its corresponding start/end position in order,
    and uses the first one that maps successfully to the PDB structure."""
    result = {}
    for pept_id in pept_list:
        if pept_id not in annotprot.index:
            continue
        row      = annotprot.loc[pept_id]
        proteins = _proteins_of(row['Protein'])
        starts   = [s.strip() for s in str(row['startPosition']).split(',')]
        ends     = [e.strip() for e in str(row['endPosition']).split(',')]
        for i, uniprot_id in enumerate(proteins):
            if uniprot_id not in sifts:
                continue
            try:
                pept_start = int(starts[i] if i < len(starts) else starts[0])
                pept_end   = int(ends[i]   if i < len(ends)   else ends[0])
            except (ValueError, TypeError, IndexError):
                continue
            pdb_positions = map_peptide_to_pdb(pept_start, pept_end, sifts[uniprot_id])
            if pdb_positions:
                result[pept_id] = pdb_positions
                break  # use first successful mapping
    return result

# Map significant peptides (used for visualization coloring)
pept_to_pdb      = map_pept_list(siglip_ribo)
pdb_sig_only     = {p: pept_to_pdb[p] for p in sig_only_ribo     if p in pept_to_pdb}
pdb_sig_corr     = {p: pept_to_pdb[p] for p in sig_and_corr_ribo if p in pept_to_pdb}

# Map ALL matched-protein peptides (needed for SASA background comparison)
all_ribo_pept_to_pdb = map_pept_list(all_ribo_pepts)

print(f'Significant peptides mapped to PDB:              {len(pept_to_pdb)}')
print(f'  Non-correlating (\u2192 {COLOR_SIG}):       {len(pdb_sig_only)}')
print(f'  Correlating     (\u2192 {COLOR_CORR}):       {len(pdb_sig_corr)}')
print(f'All matched-protein peptides mapped:             {len(all_ribo_pept_to_pdb)}')

# ------------------------------------------------------------------
# Build per-residue group sets (used for visualization and SASA)
# ------------------------------------------------------------------
def residues_from_pept_dict(pept_dict):
    """Collect all (chain, res_num) pairs covered by any peptide in the dict."""
    res_set = set()
    for positions in pept_dict.values():
        for pos in positions:
            for rn in pos['residues']:
                res_set.add((pos['chain'], rn))
    return res_set

detected_res_set = residues_from_pept_dict(all_ribo_pept_to_pdb)   # any detected peptide
sig_res_set      = residues_from_pept_dict(pept_to_pdb)             # any siglip peptide
corr_res_set     = residues_from_pept_dict(pdb_sig_corr)            # correlating peptide

# All PDB residues for matched proteins (from SIFTS) -- defines full expected coverage
sifts_residues = set()
for uniprot_id in matched_proteins:
    if uniprot_id not in sifts:
        continue
    for cm in sifts[uniprot_id]:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            sifts_residues.add((cm['chain_id'], unp_res + offset))

print(f'\nPDB residues in matched proteins (SIFTS):  {len(sifts_residues)}')
print(f'  Covered by any detected peptide:          {len(detected_res_set)}')
print(f'  Covered by significant peptide:           {len(sig_res_set)}')
print(f'  Covered by correlating peptide:           {len(corr_res_set)}')
print(f'  Not detected (no LiP-MS coverage):        {len(sifts_residues - detected_res_set)}')


Significant peptides mapped to PDB:              13
  Non-correlating (→ #ea9c8e):       11
  Correlating     (→ #d8624d):       2
All matched-protein peptides mapped:             46

PDB residues in matched proteins (SIFTS):  751
  Covered by any detected peptide:          456
  Covered by significant peptide:           145
  Covered by correlating peptide:           19
  Not detected (no LiP-MS coverage):        295


## 7. Visualization: Peptide Level

All atoms shown as spacefill (grey). Significant peptide residues highlighted by category.

In [240]:
# Load structure string (done once, reused for both views)
print(f'Loading CIF structure ({file_size_mb:.1f} MB)...')
with open(cif_path) as f:
    cif_str = f.read()
print('Done.')

Loading CIF structure (0.7 MB)...
Done.


In [241]:
view1 = py3Dmol.view(width=950, height=620)
view1.addModel(cif_str, 'cif')

# ── 1. Base style: semi-transparent cartoon for everything ───────────────────
# RNA/DNA chains will show as ribbons; protein atoms will be overridden below.
view1.setStyle({}, {'cartoon': {'color': '#444444', 'opacity': 0.5}})

# ── 2. Remove cartoon from protein chains (replaced by VDW surfaces below) ───
all_prot_chains = {cm['chain_id'] for uid in sifts for cm in sifts[uid]} - rna_chains
for chain in all_prot_chains:
    view1.setStyle({'chain': chain}, {})

# ── 3. Small molecules: black sticks (non-water HETATM) ──────────────────────
view1.setStyle({'hetflag': True}, {'stick': {'color': 'black', 'radius': 0.25}})
view1.setStyle({'hetflag': True, 'resn': 'HOH'}, {})   # hide water

# ── 4. All RNA/DNA chains: opaque black cartoon ───────────────────────────────
for chain in rna_chains:
    view1.setStyle({'chain': chain}, {'cartoon': {'color': 'black', 'opacity': 1.0}})

# ── 5. Group significant/correlating residues by chain ───────────────────────
corr_by_chain = defaultdict(set)
for chain, res in corr_res_set:
    corr_by_chain[chain].add(res)

sig_only_by_chain = defaultdict(set)
for chain, res in sig_res_set:
    if res not in corr_by_chain[chain]:
        sig_only_by_chain[chain].add(res)

# ── 6. Build per-chain full residue sets from SIFTS ──────────────────────────
chain_all_residues_sifts = defaultdict(set)
for uid, chain_maps in sifts.items():
    for cm in chain_maps:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            chain_all_residues_sifts[cm['chain_id']].add(unp_res + offset)

# ── 7. Classify protein chains: focus (UNIPROT_FILTER) vs background ─────────
# If UNIPROT_FILTER is set: chains for those proteins → COLOR_BASE (focus),
#   all others → COLOR_FILTERED (or hidden if COLOR_FILTERED is None).
# If UNIPROT_FILTER is None: all chains are focus → COLOR_BASE.
if UNIPROT_FILTER is not None:
    focus_chains      = {cm['chain_id'] for uid in UNIPROT_FILTER
                         for cm in sifts.get(uid, [])}
    background_chains = all_prot_chains - focus_chains
else:
    focus_chains      = all_prot_chains
    background_chains = set()

# ── 8. Render background chains ──────────────────────────────────────────────
if COLOR_FILTERED is not None:
    for chain in background_chains:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': COLOR_FILTERED},
                         {'chain': chain})

# ── 9. Render focus chain surfaces (COLOR_BASE + sig/corr highlights) ────────
for chain in focus_chains:
    colored = sig_only_by_chain[chain] | corr_by_chain[chain]

    if colored and chain in chain_all_residues_sifts:
        base_residues = sorted(chain_all_residues_sifts[chain] - colored)
        if base_residues:
            view1.addSurface(py3Dmol.VDW,
                             {'opacity': 0.8, 'color': COLOR_BASE},
                             {'chain': chain, 'resi': base_residues})
    else:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': COLOR_BASE},
                         {'chain': chain})

    # Significant non-correlating residues
    if sig_only_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_SIG},
                         {'chain': chain, 'resi': sorted(sig_only_by_chain[chain])})

    # Correlating residues
    if corr_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_CORR},
                         {'chain': chain, 'resi': sorted(corr_by_chain[chain])})

view1.zoomTo()
view1.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [242]:
os.makedirs('plots', exist_ok=True)
out_path1 = f'plots/{OUTPUT_PREFIX}_peptide_level.html'
view1.write_html(out_path1)
print(f'Saved: {out_path1}')

Saved: plots/MES1_peptide_level.html


# CDC60

## Parameters

Edit the cell below to re-run this analysis for a different protein complex. All downstream cells adapt automatically.

In [ ]:
# ============================================================
# PARAMETERS — edit only this cell to switch to another structure
# ============================================================

# Structure source: 'pdb' downloads from RCSB, 'alphafold' downloads from AlphaFold EBI
STRUCTURE_SOURCE = 'alphafold'

# RCSB PDB identifier (used when STRUCTURE_SOURCE = 'pdb')
PDB_ID           = None

# UniProt accession for AlphaFold (used when STRUCTURE_SOURCE = 'alphafold')
# e.g. 'P38179' — downloads AF-P38179-F1-model_v4.cif from alphafold.ebi.ac.uk
ALPHAFOLD_ID     = 'P26637'

ANALYSIS_LABEL   = 'CDC60'   # label used in plot titles
OUTPUT_PREFIX    = 'CDC60'   # prefix for all output files

# Restrict to a subset of UniProt IDs present in both the LiP-MS data and the structure.
# None = use ALL proteins found in the intersection automatically.
# For AlphaFold, leave as None (auto) or set to [ALPHAFOLD_ID].
UNIPROT_FILTER = None

# Color palette (used throughout downstream cells)
COLOR_FILTERED   = '#888888'  # chains NOT in UNIPROT_FILTER (if set); use None to hide them
COLOR_BASE       = '#888888'  # focus protein chains (all proteins if UNIPROT_FILTER is None)
COLOR_SIG        = '#ea9c8e'  # significant LiP peptides, NOT metabolite-correlated
COLOR_CORR       = '#d8624d'  # significant LiP peptides, correlating with metabolites


## 1. Load LiP-MS Data

In [121]:
# Load differential structurome results from all three nutrient conditions
diff_g = pd.read_csv('results/diffStructurome_G.tsv', sep='\t', index_col='quantID')
diff_n = pd.read_csv('results/diffStructurome_N.tsv', sep='\t', index_col='quantID')
diff_p = pd.read_csv('results/diffStructurome_P.tsv', sep='\t', index_col='quantID')

# Build annotprot: union of all conditions, deduplicated by quantID
annot_cols = ['Peptide', 'Protein', 'gene_name', 'startPosition', 'endPosition', 'isTryptic']
annotprot = pd.concat([diff_g[annot_cols], diff_n[annot_cols], diff_p[annot_cols]])
annotprot = annotprot[~annotprot.index.duplicated(keep='first')]

# siglip: peptides significant (adj.P.Val < 0.05) in at least one condition
siglip_g = set(diff_g.index[(diff_g['adj.P.Val'] < 0.05) & diff_g['adj.P.Val'].notna()])
siglip_n = set(diff_n.index[(diff_n['adj.P.Val'] < 0.05) & diff_n['adj.P.Val'].notna()])
siglip_p = set(diff_p.index[(diff_p['adj.P.Val'] < 0.05) & diff_p['adj.P.Val'].notna()])
siglip = list(siglip_g | siglip_n | siglip_p)

print(f'Total peptides in annotprot: {len(annotprot)}')
print(f'Significant peptides (siglip): {len(siglip)}')
print(f'  G: {len(siglip_g)}, N: {len(siglip_n)}, P: {len(siglip_p)}')
annotprot.head(3)

Total peptides in annotprot: 29732
Significant peptides (siglip): 5540
  G: 3204, N: 1112, P: 1782


,Peptide,Protein,gene_name,startPosition,endPosition,isTryptic
quantID,,,,,,
EMIISSLSPLEPHSQNEILQYYL,EMIISSLSPLEPHSQNEILQYYL,P12945,NAT1,832,854,Specific
SFDDLHK,SFDDLHK,P17709,GLK1,2,8,Specific-N
EQNEDLGLSDIKVE,EQNEDLGLSDIKVE,P53303,ZPR1,473,486,Specific


## 2. Load Correlation Data

In [122]:
# Load LiP-MS vs Metabolomics correlation results
# X1 = peptide (quantID), X2 = metabolite, lfdr = local false discovery rate
cors = pd.read_csv('source_data/LiPMS_vs_Metab.tsv', sep='\t', index_col=0)
print(f'Columns: {list(cors.columns)}')
print(f'Total peptide-metabolite pairs tested: {len(cors)}')

# Correlating peptides: those with lfdr < 0.05 in at least one metabolite pair
correlating_pepts = set(cors.loc[cors['lfdr'] < 0.05, 'X1'])
print(f'Peptides with significant metabolite correlations (lfdr < 0.05): {len(correlating_pepts)}')

# Classify siglip peptides into two categories:
#   sig_only     → significant LiP but NOT metabolite-correlated → color #4c7b6a
#   sig_and_corr → significant LiP AND metabolite-correlated   → color #d8624d
sig_only     = [p for p in siglip if p not in correlating_pepts]
sig_and_corr = [p for p in siglip if p in correlating_pepts]

print(f'\nSignificant, non-correlating (→ #4c7b6a): {len(sig_only)}')
print(f'Significant AND correlating  (→ #d8624d): {len(sig_and_corr)}')

Columns: ['X1', 'X2', 'pval', 'qval', 'lfdr', 'R2', 'annot_pept', 'annot_metab', 'GO_translation']
Total peptide-metabolite pairs tested: 1429866
Peptides with significant metabolite correlations (lfdr < 0.05): 1558

Significant, non-correlating (→ #4c7b6a): 3982
Significant AND correlating  (→ #d8624d): 1558


## 3. Fetch Structure and SIFTS Mapping


In [123]:
os.makedirs('data', exist_ok=True)

if STRUCTURE_SOURCE == 'alphafold':
    struct_id = f'AF-{ALPHAFOLD_ID}-F1'
    cif_path  = f'data/{struct_id}.cif'
    if not os.path.exists(cif_path):
        print(f'Downloading AlphaFold structure for {ALPHAFOLD_ID}...')
        url  = f'https://alphafold.ebi.ac.uk/files/{struct_id}-model_v6.cif'
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cif_path, 'w') as f:
            f.write(resp.text)
        print(f'Saved to {cif_path}')
    else:
        print(f'Using cached {cif_path}')
else:
    struct_id = PDB_ID
    cif_path  = f'data/{PDB_ID}.cif'
    if not os.path.exists(cif_path):
        print(f'Downloading {PDB_ID} from RCSB (may take a minute for large structures)...')
        url  = f'https://files.rcsb.org/download/{PDB_ID}.cif'
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cif_path, 'w') as f:
            f.write(resp.text)
        print(f'Saved to {cif_path}')
    else:
        print(f'Using cached {cif_path}')

file_size_mb = os.path.getsize(cif_path) / 1e6
print(f'File size: {file_size_mb:.1f} MB')


Saved to data/AF-P26637-F1.cif
File size: 1.0 MB


In [124]:
def get_sifts_mapping(pdb_id):
    """
    Fetch UniProt -> PDB chain/residue mapping from PDBe SIFTS API.
    Returns:
        sifts:      {uniprot_id: [{'chain_id', 'unp_start', 'unp_end', 'pdb_start', 'pdb_end'}]}
        sifts_ident: {uniprot_id: identifier_string}  e.g. 'RL2A_YEAST'
    """
    url = f'https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id.lower()}'
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    pdb_key = pdb_id.lower()

    sifts       = {}
    sifts_ident = {}
    for uniprot_id, info in data[pdb_key]['UniProt'].items():
        chain_maps = []
        for m in info['mappings']:
            chain_maps.append({
                'chain_id': m['chain_id'],
                'unp_start': m['unp_start'],
                'unp_end':   m['unp_end'],
                'pdb_start': m['start']['residue_number'],
                'pdb_end':   m['end']['residue_number'],
            })
        sifts[uniprot_id]       = chain_maps
        sifts_ident[uniprot_id] = info.get('identifier', '')
    return sifts, sifts_ident


if STRUCTURE_SOURCE == 'alphafold':
    # AlphaFold models always use chain 'A' with residue numbers = UniProt positions.
    # Fetch exact sequence length from UniProt REST API.
    print(f'AlphaFold mode: fetching sequence length for {ALPHAFOLD_ID} from UniProt...')
    try:
        _resp = requests.get(
            f'https://rest.uniprot.org/uniprotkb/{ALPHAFOLD_ID}.json', timeout=30)
        _resp.raise_for_status()
        _seq_len = _resp.json()['sequence']['length']
        print(f'  Sequence length: {_seq_len} aa')
    except Exception as e:
        print(f'  UniProt API failed ({e}), falling back to length=10000')
        _seq_len = 10000
    sifts = {
        ALPHAFOLD_ID: [{'chain_id': 'A',
                         'unp_start': 1,    'unp_end': _seq_len,
                         'pdb_start': 1,    'pdb_end': _seq_len}]
    }
    sifts_ident = {ALPHAFOLD_ID: ALPHAFOLD_ID}
    print(f'Synthetic SIFTS: chain A, residues 1-{_seq_len} -> {ALPHAFOLD_ID}')
else:
    print(f'Fetching SIFTS mapping for {PDB_ID}...')
    sifts, sifts_ident = get_sifts_mapping(PDB_ID)
    print(f'Proteins with SIFTS mappings in {PDB_ID}: {len(sifts)}')
    print(f'Example UniProt IDs: {list(sifts.keys())[:8]}')


AlphaFold mode: fetching sequence length for P26637 from UniProt...
  Sequence length: 1090 aa
Synthetic SIFTS: chain A, residues 1-1090 -> P26637


## 4. Identify Peptides

In [125]:
# Find which LiP-MS proteins overlap with proteins in the ribosome structure
# Handle comma-separated protein IDs (ambiguously assigned peptides)
def _proteins_of(protein_str):
    """Return list of individual UniProt IDs from a possibly comma-separated string."""
    return [p.strip() for p in str(protein_str).split(',')]

lipm_proteins     = {p for prot_str in annotprot['Protein'] for p in _proteins_of(prot_str)}
ribosome_proteins = set(sifts.keys())
matched_proteins  = lipm_proteins & ribosome_proteins

# Apply optional UniProt filter (set UNIPROT_FILTER in the Parameters cell)
if UNIPROT_FILTER is not None:
    matched_proteins = matched_proteins & set(UNIPROT_FILTER)
    print(f'Restricted to {len(UNIPROT_FILTER)} specified UniProt IDs \u2192 {len(matched_proteins)} matched')

print(f'Proteins in LiP-MS dataset:  {len(lipm_proteins)}')
print(f'Proteins in {PDB_ID}:         {len(ribosome_proteins)}')
print(f'Matched proteins:            {len(matched_proteins)}')

# Filter peptide lists to matched proteins only
def filter_to_target(pept_list):
    return [p for p in pept_list
            if p in annotprot.index and
            any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]

siglip_ribo       = filter_to_target(siglip)
sig_only_ribo     = filter_to_target(sig_only)
sig_and_corr_ribo = filter_to_target(sig_and_corr)

print(f'\nPeptides from matched proteins in siglip:      {len(siglip_ribo)}')
print(f'  Sig, non-correlating (\u2192 {COLOR_SIG}):  {len(sig_only_ribo)}')
print(f'  Sig AND correlating  (\u2192 {COLOR_CORR}):  {len(sig_and_corr_ribo)}')

# All peptides from matched proteins (background for SASA analysis)
all_ribo_pepts = [p for p in annotprot.index
                  if any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]
print(f'\nAll matched-protein peptides (SASA background): {len(all_ribo_pepts)}')

# Show matched proteins and gene names
ribo_annot = (annotprot[annotprot['Protein'].apply(
                  lambda x: any(p in matched_proteins for p in _proteins_of(x)))]
              [['Protein', 'gene_name']].drop_duplicates().sort_values('gene_name'))
print(f'\nMatched proteins ({len(ribo_annot)}):')
display(ribo_annot)


Proteins in LiP-MS dataset:  2510
Proteins in None:         1
Matched proteins:            1

Peptides from matched proteins in siglip:      15
  Sig, non-correlating (→ #ea9c8e):  13
  Sig AND correlating  (→ #d8624d):  2

All matched-protein peptides (SASA background): 53

Matched proteins (1):


,Protein,gene_name
quantID,,
LANEKPEDVFER,P26637,CDC60


## 5. Map Peptide Positions to PDB Residues

Uses SIFTS to convert UniProt sequence positions (startPosition, endPosition from LiP-MS) to PDB chain + residue numbers.

In [126]:
from collections import defaultdict

def map_peptide_to_pdb(pept_start, pept_end, chain_maps):
    """
    Map UniProt peptide [pept_start, pept_end] to PDB chain + residue list.
    Handles cases where a peptide spans multiple mapping segments.
    Returns: [{'chain': chain_id, 'residues': [res1, res2, ...]}]
    """
    results = []
    for cm in chain_maps:
        ov_start = max(pept_start, cm['unp_start'])
        ov_end   = min(pept_end,   cm['unp_end'])
        if ov_start <= ov_end:
            offset  = cm['pdb_start'] - cm['unp_start']
            pdb_res = list(range(ov_start + offset, ov_end + offset + 1))
            results.append({'chain': cm['chain_id'], 'residues': pdb_res})
    return results

def map_pept_list(pept_list):
    """Map a list of peptide IDs to PDB chain + residue positions.
    Handles peptides ambiguously assigned to multiple comma-separated proteins:
    tries each protein with its corresponding start/end position in order,
    and uses the first one that maps successfully to the PDB structure."""
    result = {}
    for pept_id in pept_list:
        if pept_id not in annotprot.index:
            continue
        row      = annotprot.loc[pept_id]
        proteins = _proteins_of(row['Protein'])
        starts   = [s.strip() for s in str(row['startPosition']).split(',')]
        ends     = [e.strip() for e in str(row['endPosition']).split(',')]
        for i, uniprot_id in enumerate(proteins):
            if uniprot_id not in sifts:
                continue
            try:
                pept_start = int(starts[i] if i < len(starts) else starts[0])
                pept_end   = int(ends[i]   if i < len(ends)   else ends[0])
            except (ValueError, TypeError, IndexError):
                continue
            pdb_positions = map_peptide_to_pdb(pept_start, pept_end, sifts[uniprot_id])
            if pdb_positions:
                result[pept_id] = pdb_positions
                break  # use first successful mapping
    return result

# Map significant peptides (used for visualization coloring)
pept_to_pdb      = map_pept_list(siglip_ribo)
pdb_sig_only     = {p: pept_to_pdb[p] for p in sig_only_ribo     if p in pept_to_pdb}
pdb_sig_corr     = {p: pept_to_pdb[p] for p in sig_and_corr_ribo if p in pept_to_pdb}

# Map ALL matched-protein peptides (needed for SASA background comparison)
all_ribo_pept_to_pdb = map_pept_list(all_ribo_pepts)

print(f'Significant peptides mapped to PDB:              {len(pept_to_pdb)}')
print(f'  Non-correlating (\u2192 {COLOR_SIG}):       {len(pdb_sig_only)}')
print(f'  Correlating     (\u2192 {COLOR_CORR}):       {len(pdb_sig_corr)}')
print(f'All matched-protein peptides mapped:             {len(all_ribo_pept_to_pdb)}')

# ------------------------------------------------------------------
# Build per-residue group sets (used for visualization and SASA)
# ------------------------------------------------------------------
def residues_from_pept_dict(pept_dict):
    """Collect all (chain, res_num) pairs covered by any peptide in the dict."""
    res_set = set()
    for positions in pept_dict.values():
        for pos in positions:
            for rn in pos['residues']:
                res_set.add((pos['chain'], rn))
    return res_set

detected_res_set = residues_from_pept_dict(all_ribo_pept_to_pdb)   # any detected peptide
sig_res_set      = residues_from_pept_dict(pept_to_pdb)             # any siglip peptide
corr_res_set     = residues_from_pept_dict(pdb_sig_corr)            # correlating peptide

# All PDB residues for matched proteins (from SIFTS) -- defines full expected coverage
sifts_residues = set()
for uniprot_id in matched_proteins:
    if uniprot_id not in sifts:
        continue
    for cm in sifts[uniprot_id]:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            sifts_residues.add((cm['chain_id'], unp_res + offset))

print(f'\nPDB residues in matched proteins (SIFTS):  {len(sifts_residues)}')
print(f'  Covered by any detected peptide:          {len(detected_res_set)}')
print(f'  Covered by significant peptide:           {len(sig_res_set)}')
print(f'  Covered by correlating peptide:           {len(corr_res_set)}')
print(f'  Not detected (no LiP-MS coverage):        {len(sifts_residues - detected_res_set)}')


Significant peptides mapped to PDB:              15
  Non-correlating (→ #ea9c8e):       13
  Correlating     (→ #d8624d):       2
All matched-protein peptides mapped:             53

PDB residues in matched proteins (SIFTS):  1090
  Covered by any detected peptide:          510
  Covered by significant peptide:           191
  Covered by correlating peptide:           27
  Not detected (no LiP-MS coverage):        580


## 7. Visualization: Peptide Level

All atoms shown as spacefill (grey). Significant peptide residues highlighted by category.

In [127]:
# Load structure string (done once, reused for both views)
print(f'Loading CIF structure ({file_size_mb:.1f} MB)...')
with open(cif_path) as f:
    cif_str = f.read()
print('Done.')

Loading CIF structure (1.0 MB)...
Done.


In [128]:
import re as _re

def _classify_chains(cif_text):
    '''Identify all RNA/DNA polymer chains in a mmCIF file.

    Covers rRNA, tRNA, mRNA, snRNA, DNA, and any other nucleic acid chain.

    Step 1: find RNA/DNA entity IDs from _entity description + _entity_poly type.
    Step 2: map entity_id -> label_asym_id via _struct_asym.
    Step 3: map label_asym_id -> auth_asym_id via _atom_site.

    Returns:
        rna_chains (set): auth_asym_ids for all RNA/DNA polymer chains.
    '''
    cut  = cif_text.find('\n_atom_site.')
    if cut < 0:
        cut = cif_text.find('\nloop_\n_atom_site.')
    meta = cif_text[:cut] if cut > 0 else cif_text[:500_000]

    # ── Step 1: entity IDs for RNA/DNA ───────────────────────────────────────
    rna_eids = set()

    # Unquoted entity type column (e.g. "1  1  polymer  tRNA")
    for m in _re.finditer(
            r'^\s*(\d+)\s+\S+\s+\S+\s+(?:tRNA|rRNA|mRNA|snRNA|snoRNA|RNA|DNA)\b',
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Single-quoted descriptions
    for m in _re.finditer(
            r"^\s*(\d+)\s+\S+\s+\S+\s+'([^']*(?:RNA|DNA|transfer|ribosomal|messenger)[^']*)'",
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Double-quoted descriptions
    for m in _re.finditer(
            r'^\s*(\d+)\s+\S+\s+\S+\s+"([^"]*(?:RNA|DNA|transfer|ribosomal|messenger)[^"]*)"',
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Also check _entity_poly.type for polyribonucleotide / polydeoxyribonucleotide
    ep_idx = meta.find('_entity_poly.')
    if ep_idx >= 0:
        blk_start = meta.rfind('loop_', 0, ep_idx)
        blk_end   = meta.find('\nloop_', ep_idx + 1)
        blk_end   = blk_end if blk_end > 0 else len(meta)
        blk       = meta[blk_start:blk_end]
        ep_hdrs   = _re.findall(r'_entity_poly\.(\S+)', blk)
        if 'entity_id' in ep_hdrs and 'type' in ep_hdrs:
            eid_col  = ep_hdrs.index('entity_id')
            type_col = ep_hdrs.index('type')
            last_hdr = f'_entity_poly.{ep_hdrs[-1]}'
            ds       = blk.find(last_hdr) + len(last_hdr)
            for line in blk[ds:].split('\n'):
                parts = line.split()
                if len(parts) > max(eid_col, type_col):
                    ptype = parts[type_col].lower()
                    if 'ribo' in ptype or 'deoxyribo' in ptype or 'dna' in ptype:
                        rna_eids.add(parts[eid_col])

    print(f'  RNA/DNA entity IDs: {rna_eids}')
    if not rna_eids:
        return set()

    # ── Step 2: label_asym_id for RNA entities ───────────────────────────────
    label_ids  = set()
    search_pos = 0
    while True:
        sa_idx = meta.find('_struct_asym.id', search_pos)
        if sa_idx < 0:
            break
        blk_start = meta.rfind('loop_', 0, sa_idx)
        blk_end   = meta.find('\nloop_', sa_idx)
        blk_end   = blk_end if blk_end > 0 else len(meta)
        blk       = meta[blk_start:blk_end]
        headers   = _re.findall(r'_struct_asym\.(\S+)', blk)
        try:
            id_col  = headers.index('id')
            eid_col = headers.index('entity_id')
        except ValueError:
            search_pos = sa_idx + 1
            continue
        last_hdr   = f'_struct_asym.{headers[-1]}'
        data_start = blk.find(last_hdr) + len(last_hdr)
        for line in blk[data_start:].split('\n'):
            parts = line.split()
            if len(parts) > max(id_col, eid_col) and parts[eid_col] in rna_eids:
                label_ids.add(parts[id_col])
        search_pos = sa_idx + 1

    print(f'  RNA/DNA label_asym_ids: {label_ids}')
    if not label_ids:
        return set()

    # ── Step 3: map label_asym_id -> auth_asym_id via _atom_site ─────────────
    atom_start = cif_text.find('\nloop_\n_atom_site.')
    if atom_start < 0:
        atom_start = cif_text.find('\n_atom_site.')
    hdr_block  = cif_text[atom_start:atom_start + 5000]
    hdr_names  = _re.findall(r'_atom_site\.(\S+)', hdr_block)
    try:
        label_col = hdr_names.index('label_asym_id')
        auth_col  = hdr_names.index('auth_asym_id')
    except ValueError:
        print('  WARNING: could not find label_asym_id/auth_asym_id columns; returning label IDs')
        return label_ids  # fallback

    last_hdr_str  = f'_atom_site.{hdr_names[-1]}'
    data_off      = cif_text.find(last_hdr_str, atom_start) + len(last_hdr_str)
    label_to_auth = {}
    remaining     = label_ids.copy()
    for line in cif_text[data_off:].split('\n'):
        if not remaining:
            break
        parts = line.split()
        if len(parts) <= max(label_col, auth_col):
            continue
        lbl = parts[label_col]
        if lbl in remaining:
            label_to_auth[lbl] = parts[auth_col]
            remaining.discard(lbl)

    auth_ids = set(label_to_auth.values())
    print(f'  RNA/DNA auth_asym_ids (for py3Dmol): {sorted(auth_ids)}')
    return auth_ids


rna_chains = _classify_chains(cif_str)
print(f'Detected RNA/DNA chains: {sorted(rna_chains) if rna_chains else "none found"}')


  RNA/DNA entity IDs: set()
Detected RNA/DNA chains: none found


In [129]:
view1 = py3Dmol.view(width=950, height=620)
view1.addModel(cif_str, 'cif')

# ── 1. Base style: semi-transparent cartoon for everything ───────────────────
# RNA/DNA chains will show as ribbons; protein atoms will be overridden below.
view1.setStyle({}, {'cartoon': {'color': '#444444', 'opacity': 0.5}})

# ── 2. Remove cartoon from protein chains (replaced by VDW surfaces below) ───
all_prot_chains = {cm['chain_id'] for uid in sifts for cm in sifts[uid]} - rna_chains
for chain in all_prot_chains:
    view1.setStyle({'chain': chain}, {})

# ── 3. Small molecules: black sticks (non-water HETATM) ──────────────────────
view1.setStyle({'hetflag': True}, {'stick': {'color': 'black', 'radius': 0.25}})
view1.setStyle({'hetflag': True, 'resn': 'HOH'}, {})   # hide water

# ── 4. All RNA/DNA chains: opaque black cartoon ───────────────────────────────
for chain in rna_chains:
    view1.setStyle({'chain': chain}, {'cartoon': {'color': 'black', 'opacity': 1.0}})

# ── 5. Group significant/correlating residues by chain ───────────────────────
corr_by_chain = defaultdict(set)
for chain, res in corr_res_set:
    corr_by_chain[chain].add(res)

sig_only_by_chain = defaultdict(set)
for chain, res in sig_res_set:
    if res not in corr_by_chain[chain]:
        sig_only_by_chain[chain].add(res)

# ── 6. Build per-chain full residue sets from SIFTS ──────────────────────────
chain_all_residues_sifts = defaultdict(set)
for uid, chain_maps in sifts.items():
    for cm in chain_maps:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            chain_all_residues_sifts[cm['chain_id']].add(unp_res + offset)

# ── 7. Classify protein chains: focus (UNIPROT_FILTER) vs background ─────────
# If UNIPROT_FILTER is set: chains for those proteins → COLOR_BASE (focus),
#   all others → COLOR_FILTERED (or hidden if COLOR_FILTERED is None).
# If UNIPROT_FILTER is None: all chains are focus → COLOR_BASE.
if UNIPROT_FILTER is not None:
    focus_chains      = {cm['chain_id'] for uid in UNIPROT_FILTER
                         for cm in sifts.get(uid, [])}
    background_chains = all_prot_chains - focus_chains
else:
    focus_chains      = all_prot_chains
    background_chains = set()

# ── 8. Render background chains ──────────────────────────────────────────────
if COLOR_FILTERED is not None:
    for chain in background_chains:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': COLOR_FILTERED},
                         {'chain': chain})

# ── 9. Render focus chain surfaces (COLOR_BASE + sig/corr highlights) ────────
for chain in focus_chains:
    colored = sig_only_by_chain[chain] | corr_by_chain[chain]

    if colored and chain in chain_all_residues_sifts:
        base_residues = sorted(chain_all_residues_sifts[chain] - colored)
        if base_residues:
            view1.addSurface(py3Dmol.VDW,
                             {'opacity': 0.8, 'color': COLOR_BASE},
                             {'chain': chain, 'resi': base_residues})
    else:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': COLOR_BASE},
                         {'chain': chain})

    # Significant non-correlating residues
    if sig_only_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_SIG},
                         {'chain': chain, 'resi': sorted(sig_only_by_chain[chain])})

    # Correlating residues
    if corr_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_CORR},
                         {'chain': chain, 'resi': sorted(corr_by_chain[chain])})

view1.zoomTo()
view1.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [207]:
os.makedirs('plots', exist_ok=True)
out_path1 = f'plots/{OUTPUT_PREFIX}_peptide_level.html'
view1.write_html(out_path1)
print(f'Saved: {out_path1}')

Saved: plots/MES1_peptide_level.html


# STM1

## Parameters

Edit the cell below to re-run this analysis for a different protein complex. All downstream cells adapt automatically.

In [221]:
# ============================================================
# PARAMETERS — edit only this cell to switch to another structure
# ============================================================

# Structure source: 'pdb' downloads from RCSB, 'alphafold' downloads from AlphaFold EBI
STRUCTURE_SOURCE = 'alphafold'

# RCSB PDB identifier (used when STRUCTURE_SOURCE = 'pdb')
PDB_ID           = None

# UniProt accession for AlphaFold (used when STRUCTURE_SOURCE = 'alphafold')
# e.g. 'P38179' — downloads AF-P38179-F1-model_v4.cif from alphafold.ebi.ac.uk
ALPHAFOLD_ID     = 'P39015'

ANALYSIS_LABEL   = 'STM1'   # label used in plot titles
OUTPUT_PREFIX    = 'STM1'   # prefix for all output files

# Restrict to a subset of UniProt IDs present in both the LiP-MS data and the structure.
# None = use ALL proteins found in the intersection automatically.
# For AlphaFold, leave as None (auto) or set to [ALPHAFOLD_ID].
UNIPROT_FILTER = None

# Color palette (used throughout downstream cells)
COLOR_FILTERED   = None  # chains NOT in UNIPROT_FILTER (if set); use None to hide them
COLOR_BASE       = '#888888'  # focus protein chains (all proteins if UNIPROT_FILTER is None)
COLOR_SIG        = '#ea9c8e'  # significant LiP peptides, NOT metabolite-correlated
COLOR_CORR       = '#d8624d'  # significant LiP peptides, correlating with metabolites


## 1. Load LiP-MS Data

In [222]:
# Load differential structurome results from all three nutrient conditions
diff_g = pd.read_csv('results/diffStructurome_G.tsv', sep='\t', index_col='quantID')
diff_n = pd.read_csv('results/diffStructurome_N.tsv', sep='\t', index_col='quantID')
diff_p = pd.read_csv('results/diffStructurome_P.tsv', sep='\t', index_col='quantID')

# Build annotprot: union of all conditions, deduplicated by quantID
annot_cols = ['Peptide', 'Protein', 'gene_name', 'startPosition', 'endPosition', 'isTryptic']
annotprot = pd.concat([diff_g[annot_cols], diff_n[annot_cols], diff_p[annot_cols]])
annotprot = annotprot[~annotprot.index.duplicated(keep='first')]

# siglip: peptides significant (adj.P.Val < 0.05) in at least one condition
siglip_g = set(diff_g.index[(diff_g['adj.P.Val'] < 0.05) & diff_g['adj.P.Val'].notna()])
siglip_n = set(diff_n.index[(diff_n['adj.P.Val'] < 0.05) & diff_n['adj.P.Val'].notna()])
siglip_p = set(diff_p.index[(diff_p['adj.P.Val'] < 0.05) & diff_p['adj.P.Val'].notna()])
siglip = list(siglip_g | siglip_n | siglip_p)

print(f'Total peptides in annotprot: {len(annotprot)}')
print(f'Significant peptides (siglip): {len(siglip)}')
print(f'  G: {len(siglip_g)}, N: {len(siglip_n)}, P: {len(siglip_p)}')
annotprot.head(3)

Total peptides in annotprot: 29732
Significant peptides (siglip): 5540
  G: 3204, N: 1112, P: 1782


,Peptide,Protein,gene_name,startPosition,endPosition,isTryptic
quantID,,,,,,
EMIISSLSPLEPHSQNEILQYYL,EMIISSLSPLEPHSQNEILQYYL,P12945,NAT1,832,854,Specific
SFDDLHK,SFDDLHK,P17709,GLK1,2,8,Specific-N
EQNEDLGLSDIKVE,EQNEDLGLSDIKVE,P53303,ZPR1,473,486,Specific


## 2. Load Correlation Data

In [223]:
# Load LiP-MS vs Metabolomics correlation results
# X1 = peptide (quantID), X2 = metabolite, lfdr = local false discovery rate
cors = pd.read_csv('source_data/LiPMS_vs_Metab.tsv', sep='\t', index_col=0)
print(f'Columns: {list(cors.columns)}')
print(f'Total peptide-metabolite pairs tested: {len(cors)}')

# Correlating peptides: those with lfdr < 0.05 in at least one metabolite pair
correlating_pepts = set(cors.loc[cors['lfdr'] < 0.05, 'X1'])
print(f'Peptides with significant metabolite correlations (lfdr < 0.05): {len(correlating_pepts)}')

# Classify siglip peptides into two categories:
#   sig_only     → significant LiP but NOT metabolite-correlated → color #4c7b6a
#   sig_and_corr → significant LiP AND metabolite-correlated   → color #d8624d
sig_only     = [p for p in siglip if p not in correlating_pepts]
sig_and_corr = [p for p in siglip if p in correlating_pepts]

print(f'\nSignificant, non-correlating (→ #4c7b6a): {len(sig_only)}')
print(f'Significant AND correlating  (→ #d8624d): {len(sig_and_corr)}')

Columns: ['X1', 'X2', 'pval', 'qval', 'lfdr', 'R2', 'annot_pept', 'annot_metab', 'GO_translation']
Total peptide-metabolite pairs tested: 1429866
Peptides with significant metabolite correlations (lfdr < 0.05): 1558

Significant, non-correlating (→ #4c7b6a): 3982
Significant AND correlating  (→ #d8624d): 1558


## 3. Fetch Structure and SIFTS Mapping


In [224]:
os.makedirs('data', exist_ok=True)

if STRUCTURE_SOURCE == 'alphafold':
    struct_id = f'AF-{ALPHAFOLD_ID}-F1'
    cif_path  = f'data/{struct_id}.cif'
    if not os.path.exists(cif_path):
        print(f'Downloading AlphaFold structure for {ALPHAFOLD_ID}...')
        url  = f'https://alphafold.ebi.ac.uk/files/{struct_id}-model_v6.cif'
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cif_path, 'w') as f:
            f.write(resp.text)
        print(f'Saved to {cif_path}')
    else:
        print(f'Using cached {cif_path}')
else:
    struct_id = PDB_ID
    cif_path  = f'data/{PDB_ID}.cif'
    if not os.path.exists(cif_path):
        print(f'Downloading {PDB_ID} from RCSB (may take a minute for large structures)...')
        url  = f'https://files.rcsb.org/download/{PDB_ID}.cif'
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cif_path, 'w') as f:
            f.write(resp.text)
        print(f'Saved to {cif_path}')
    else:
        print(f'Using cached {cif_path}')

file_size_mb = os.path.getsize(cif_path) / 1e6
print(f'File size: {file_size_mb:.1f} MB')


Saved to data/AF-P39015-F1.cif
File size: 0.3 MB


In [225]:
def get_sifts_mapping(pdb_id):
    """
    Fetch UniProt -> PDB chain/residue mapping from PDBe SIFTS API.
    Returns:
        sifts:      {uniprot_id: [{'chain_id', 'unp_start', 'unp_end', 'pdb_start', 'pdb_end'}]}
        sifts_ident: {uniprot_id: identifier_string}  e.g. 'RL2A_YEAST'
    """
    url = f'https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id.lower()}'
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    pdb_key = pdb_id.lower()

    sifts       = {}
    sifts_ident = {}
    for uniprot_id, info in data[pdb_key]['UniProt'].items():
        chain_maps = []
        for m in info['mappings']:
            chain_maps.append({
                'chain_id': m['chain_id'],
                'unp_start': m['unp_start'],
                'unp_end':   m['unp_end'],
                'pdb_start': m['start']['residue_number'],
                'pdb_end':   m['end']['residue_number'],
            })
        sifts[uniprot_id]       = chain_maps
        sifts_ident[uniprot_id] = info.get('identifier', '')
    return sifts, sifts_ident


if STRUCTURE_SOURCE == 'alphafold':
    # AlphaFold models always use chain 'A' with residue numbers = UniProt positions.
    # Fetch exact sequence length from UniProt REST API.
    print(f'AlphaFold mode: fetching sequence length for {ALPHAFOLD_ID} from UniProt...')
    try:
        _resp = requests.get(
            f'https://rest.uniprot.org/uniprotkb/{ALPHAFOLD_ID}.json', timeout=30)
        _resp.raise_for_status()
        _seq_len = _resp.json()['sequence']['length']
        print(f'  Sequence length: {_seq_len} aa')
    except Exception as e:
        print(f'  UniProt API failed ({e}), falling back to length=10000')
        _seq_len = 10000
    sifts = {
        ALPHAFOLD_ID: [{'chain_id': 'A',
                         'unp_start': 1,    'unp_end': _seq_len,
                         'pdb_start': 1,    'pdb_end': _seq_len}]
    }
    sifts_ident = {ALPHAFOLD_ID: ALPHAFOLD_ID}
    print(f'Synthetic SIFTS: chain A, residues 1-{_seq_len} -> {ALPHAFOLD_ID}')
else:
    print(f'Fetching SIFTS mapping for {PDB_ID}...')
    sifts, sifts_ident = get_sifts_mapping(PDB_ID)
    print(f'Proteins with SIFTS mappings in {PDB_ID}: {len(sifts)}')
    print(f'Example UniProt IDs: {list(sifts.keys())[:8]}')


AlphaFold mode: fetching sequence length for P39015 from UniProt...
  Sequence length: 273 aa
Synthetic SIFTS: chain A, residues 1-273 -> P39015


## 4. Identify Peptides

In [226]:
# Find which LiP-MS proteins overlap with proteins in the ribosome structure
# Handle comma-separated protein IDs (ambiguously assigned peptides)
def _proteins_of(protein_str):
    """Return list of individual UniProt IDs from a possibly comma-separated string."""
    return [p.strip() for p in str(protein_str).split(',')]

lipm_proteins     = {p for prot_str in annotprot['Protein'] for p in _proteins_of(prot_str)}
ribosome_proteins = set(sifts.keys())
matched_proteins  = lipm_proteins & ribosome_proteins

# Apply optional UniProt filter (set UNIPROT_FILTER in the Parameters cell)
if UNIPROT_FILTER is not None:
    matched_proteins = matched_proteins & set(UNIPROT_FILTER)
    print(f'Restricted to {len(UNIPROT_FILTER)} specified UniProt IDs \u2192 {len(matched_proteins)} matched')

print(f'Proteins in LiP-MS dataset:  {len(lipm_proteins)}')
print(f'Proteins in {PDB_ID}:         {len(ribosome_proteins)}')
print(f'Matched proteins:            {len(matched_proteins)}')

# Filter peptide lists to matched proteins only
def filter_to_target(pept_list):
    return [p for p in pept_list
            if p in annotprot.index and
            any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]

siglip_ribo       = filter_to_target(siglip)
sig_only_ribo     = filter_to_target(sig_only)
sig_and_corr_ribo = filter_to_target(sig_and_corr)

print(f'\nPeptides from matched proteins in siglip:      {len(siglip_ribo)}')
print(f'  Sig, non-correlating (\u2192 {COLOR_SIG}):  {len(sig_only_ribo)}')
print(f'  Sig AND correlating  (\u2192 {COLOR_CORR}):  {len(sig_and_corr_ribo)}')

# All peptides from matched proteins (background for SASA analysis)
all_ribo_pepts = [p for p in annotprot.index
                  if any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]
print(f'\nAll matched-protein peptides (SASA background): {len(all_ribo_pepts)}')

# Show matched proteins and gene names
ribo_annot = (annotprot[annotprot['Protein'].apply(
                  lambda x: any(p in matched_proteins for p in _proteins_of(x)))]
              [['Protein', 'gene_name']].drop_duplicates().sort_values('gene_name'))
print(f'\nMatched proteins ({len(ribo_annot)}):')
display(ribo_annot)


Proteins in LiP-MS dataset:  2510
Proteins in None:         1
Matched proteins:            1

Peptides from matched proteins in siglip:      10
  Sig, non-correlating (→ #ea9c8e):  4
  Sig AND correlating  (→ #d8624d):  6

All matched-protein peptides (SASA background): 33

Matched proteins (1):


,Protein,gene_name
quantID,,
NIDVSNLPSLA,P39015,STM1


## 5. Map Peptide Positions to PDB Residues

Uses SIFTS to convert UniProt sequence positions (startPosition, endPosition from LiP-MS) to PDB chain + residue numbers.

In [227]:
from collections import defaultdict

def map_peptide_to_pdb(pept_start, pept_end, chain_maps):
    """
    Map UniProt peptide [pept_start, pept_end] to PDB chain + residue list.
    Handles cases where a peptide spans multiple mapping segments.
    Returns: [{'chain': chain_id, 'residues': [res1, res2, ...]}]
    """
    results = []
    for cm in chain_maps:
        ov_start = max(pept_start, cm['unp_start'])
        ov_end   = min(pept_end,   cm['unp_end'])
        if ov_start <= ov_end:
            offset  = cm['pdb_start'] - cm['unp_start']
            pdb_res = list(range(ov_start + offset, ov_end + offset + 1))
            results.append({'chain': cm['chain_id'], 'residues': pdb_res})
    return results

def map_pept_list(pept_list):
    """Map a list of peptide IDs to PDB chain + residue positions.
    Handles peptides ambiguously assigned to multiple comma-separated proteins:
    tries each protein with its corresponding start/end position in order,
    and uses the first one that maps successfully to the PDB structure."""
    result = {}
    for pept_id in pept_list:
        if pept_id not in annotprot.index:
            continue
        row      = annotprot.loc[pept_id]
        proteins = _proteins_of(row['Protein'])
        starts   = [s.strip() for s in str(row['startPosition']).split(',')]
        ends     = [e.strip() for e in str(row['endPosition']).split(',')]
        for i, uniprot_id in enumerate(proteins):
            if uniprot_id not in sifts:
                continue
            try:
                pept_start = int(starts[i] if i < len(starts) else starts[0])
                pept_end   = int(ends[i]   if i < len(ends)   else ends[0])
            except (ValueError, TypeError, IndexError):
                continue
            pdb_positions = map_peptide_to_pdb(pept_start, pept_end, sifts[uniprot_id])
            if pdb_positions:
                result[pept_id] = pdb_positions
                break  # use first successful mapping
    return result

# Map significant peptides (used for visualization coloring)
pept_to_pdb      = map_pept_list(siglip_ribo)
pdb_sig_only     = {p: pept_to_pdb[p] for p in sig_only_ribo     if p in pept_to_pdb}
pdb_sig_corr     = {p: pept_to_pdb[p] for p in sig_and_corr_ribo if p in pept_to_pdb}

# Map ALL matched-protein peptides (needed for SASA background comparison)
all_ribo_pept_to_pdb = map_pept_list(all_ribo_pepts)

print(f'Significant peptides mapped to PDB:              {len(pept_to_pdb)}')
print(f'  Non-correlating (\u2192 {COLOR_SIG}):       {len(pdb_sig_only)}')
print(f'  Correlating     (\u2192 {COLOR_CORR}):       {len(pdb_sig_corr)}')
print(f'All matched-protein peptides mapped:             {len(all_ribo_pept_to_pdb)}')

# ------------------------------------------------------------------
# Build per-residue group sets (used for visualization and SASA)
# ------------------------------------------------------------------
def residues_from_pept_dict(pept_dict):
    """Collect all (chain, res_num) pairs covered by any peptide in the dict."""
    res_set = set()
    for positions in pept_dict.values():
        for pos in positions:
            for rn in pos['residues']:
                res_set.add((pos['chain'], rn))
    return res_set

detected_res_set = residues_from_pept_dict(all_ribo_pept_to_pdb)   # any detected peptide
sig_res_set      = residues_from_pept_dict(pept_to_pdb)             # any siglip peptide
corr_res_set     = residues_from_pept_dict(pdb_sig_corr)            # correlating peptide

# All PDB residues for matched proteins (from SIFTS) -- defines full expected coverage
sifts_residues = set()
for uniprot_id in matched_proteins:
    if uniprot_id not in sifts:
        continue
    for cm in sifts[uniprot_id]:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            sifts_residues.add((cm['chain_id'], unp_res + offset))

print(f'\nPDB residues in matched proteins (SIFTS):  {len(sifts_residues)}')
print(f'  Covered by any detected peptide:          {len(detected_res_set)}')
print(f'  Covered by significant peptide:           {len(sig_res_set)}')
print(f'  Covered by correlating peptide:           {len(corr_res_set)}')
print(f'  Not detected (no LiP-MS coverage):        {len(sifts_residues - detected_res_set)}')


Significant peptides mapped to PDB:              10
  Non-correlating (→ #ea9c8e):       4
  Correlating     (→ #d8624d):       6
All matched-protein peptides mapped:             33

PDB residues in matched proteins (SIFTS):  273
  Covered by any detected peptide:          155
  Covered by significant peptide:           103
  Covered by correlating peptide:           54
  Not detected (no LiP-MS coverage):        118


## 6. Solvent Accessibility Analysis

Computes per-residue **relative SASA (rSASA)** on the full complex using freesasa, then compares rSASA distributions across four groups (one data point = one residue, assigned to highest-priority group):

Results are cached to `data/<PDB_ID>_residue_sasa.pkl` after the first run.

In [228]:
sasa_cache_path = f'data/{struct_id}_residue_sasa.pkl'

if os.path.exists(sasa_cache_path):
    print(f'Loading cached SASA from {sasa_cache_path}...')
    with open(sasa_cache_path, 'rb') as f:
        res_sasa = pickle.load(f)
    print(f'Loaded rSASA for {len(res_sasa)} residues.')
else:
    import gemmi

    # ── 1. Parse CIF with gemmi (fast C++ library, seconds not hours) ──────────
    print(f'Parsing {PDB_ID} CIF with gemmi...')
    st    = gemmi.read_structure(cif_path)
    model = st[0]

    # ── 2. PDB format only allows 1-char chain IDs.  Remap any multi-char     ──
    #       auth_asym_ids and keep a reverse lookup so we can restore them later.
    single_used = {ch.name for ch in model if len(ch.name) == 1}
    spare       = [c for c in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789'
                   if c not in single_used]
    pdb_to_auth = {ch.name: ch.name for ch in model if len(ch.name) == 1}
    spare_idx   = 0
    for chain in model:
        if len(chain.name) > 1:
            orig          = chain.name
            new_id        = spare[spare_idx] if spare_idx < len(spare) else '?'
            spare_idx    += 1
            pdb_to_auth[new_id] = orig
            chain.name    = new_id   # rename in-place for PDB output

    print(f'  {len(list(model))} chains; {spare_idx} multi-char chain IDs remapped for PDB format')

    # ── 3. Write temporary PDB and run freesasa ────────────────────────────────
    tmp_pdb = cif_path.replace('.cif', '_freesasa_tmp.pdb')
    st.write_pdb(tmp_pdb)
    print(f'  Temporary PDB written ({os.path.getsize(tmp_pdb)/1e6:.0f} MB)')

    print('Computing SASA with freesasa...')
    try:
        fs_structure = freesasa.Structure(tmp_pdb)
        sasa_result  = freesasa.calc(fs_structure)
    finally:
        os.remove(tmp_pdb)

    # ── 4. Build per-residue rSASA lookup with original auth_asym_id labels ───
    res_sasa = {}
    for pdb_chain, chain_areas in sasa_result.residueAreas().items():
        auth_chain = pdb_to_auth.get(pdb_chain, pdb_chain)
        for res_label, res_area in chain_areas.items():
            try:
                res_num = int(str(res_label).strip())
                rsa     = res_area.relativeTotal
                if rsa is not None:
                    res_sasa[(auth_chain, res_num)] = float(rsa)
            except (ValueError, TypeError):
                pass

    with open(sasa_cache_path, 'wb') as f:
        pickle.dump(res_sasa, f)
    print(f'SASA cached to {sasa_cache_path}')

n_valid = sum(1 for v in res_sasa.values() if not np.isnan(v))
print(f'Residues with valid rSASA: {n_valid}')


Parsing None CIF with gemmi...
  1 chains; 0 multi-char chain IDs remapped for PDB format
  Temporary PDB written (0 MB)
Computing SASA with freesasa...
SASA cached to data/AF-P39015-F1_residue_sasa.pkl
Residues with valid rSASA: 273


In [229]:
def assign_residue_group(res_key):
    if res_key in corr_res_set:
        return 'correlating'
    elif res_key in sig_res_set:
        return 'significant'
    elif res_key in detected_res_set:
        return 'detected, non-sig'
    else:
        return 'not detected'

# Build per-residue rSASA table
sasa_rows = []
for res_key in sifts_residues:
    if res_key not in res_sasa:
        continue
    rsa = res_sasa[res_key]
    if np.isnan(rsa):
        continue
    sasa_rows.append({'chain': res_key[0], 'res_num': res_key[1],
                      'lip_group': assign_residue_group(res_key), 'rSASA': rsa})

sasa_df = pd.DataFrame(sasa_rows)

group_order  = ['not detected', 'detected, non-sig', 'significant', 'correlating']
group_labels = ['Not\ndetected', 'Detected,\nnon-sig', 'Significant\n(all)', 'Correlating\nw/ metabolites']
group_colors = {
    'not detected':      COLOR_FILTERED,
    'detected, non-sig': COLOR_BASE,
    'significant':       COLOR_SIG,
    'correlating':       COLOR_CORR,
}

sasa_df['lip_group'] = pd.Categorical(sasa_df['lip_group'], categories=group_order, ordered=True)

print('Group sizes (residues):')
print(sasa_df['lip_group'].value_counts().reindex(group_order).to_string())

# ── Statistical tests (one-tailed MWU: fg > ref) ─────────────────────────────
def mwu_test(fg_vals, ref_vals):
    _, pval = stats.mannwhitneyu(fg_vals, ref_vals, alternative='greater')
    stars = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    return stars, pval

def grp_vals(grp):
    return sasa_df.loc[sasa_df['lip_group'] == grp, 'rSASA'].values

nd  = grp_vals('not detected')
dns = grp_vals('detected, non-sig')

# Set 1: all groups vs not detected; Set 2: sig/corr vs detected non-sig
test_pairs = [
    ('not detected',      'detected, non-sig', nd),
    ('not detected',      'significant',       nd),
    ('not detected',      'correlating',       nd),
    ('detected, non-sig', 'significant',       dns),
    ('detected, non-sig', 'correlating',       dns),
]
tests = []
for ref_grp, fg_grp, ref_vals in test_pairs:
    fg_vals = grp_vals(fg_grp)
    if len(fg_vals) >= 3 and len(ref_vals) >= 3:
        stars, pval = mwu_test(fg_vals, ref_vals)
        tests.append({'ref': ref_grp, 'fg': fg_grp, 'stars': stars,
                      'pval': pval})

# ── Bracket annotations ───────────────────────────────────────────────────────
y_max  = float(sasa_df['rSASA'].max())
y_step = y_max * 0.09
x_pos  = {g: i + 1 for i, g in enumerate(group_order)}
tick_h = y_step * 0.25

ref_bases = {'not detected': y_max * 1.05, 'detected, non-sig': y_max * 1.42}
ref_k     = {'not detected': 0,            'detected, non-sig': 0}

segs, txts = [], []
for t in tests:
    x1   = x_pos[t['ref']]
    x2   = x_pos[t['fg']]
    k    = ref_k[t['ref']]
    y_br = ref_bases[t['ref']] + k * y_step
    ref_k[t['ref']] += 1
    segs += [
        {'x': x1, 'xend': x2, 'y': y_br,          'yend': y_br},
        {'x': x1, 'xend': x1, 'y': y_br - tick_h, 'yend': y_br},
        {'x': x2, 'xend': x2, 'y': y_br - tick_h, 'yend': y_br},
    ]
    txts.append({
        'x':     (x1 + x2) / 2,
        'y':     y_br + y_step * 0.08,
        'label': f'{t["stars"]} p={t["pval"]:.1e}',
    })

seg_df = pd.DataFrame(segs)
txt_df = pd.DataFrame(txts)
y_lim  = ref_bases['detected, non-sig'] + ref_k['detected, non-sig'] * y_step + y_step

# ── plotnine figure ───────────────────────────────────────────────────────────
p = (
    ggplot(sasa_df, aes(x='lip_group', y='rSASA', fill='lip_group'))
    + geom_boxplot(outlier_shape='', width=0.5, show_legend=False)
    + geom_jitter(width=0.2, alpha=0.2, size=0.5,
                  color='#333333', show_legend=False)
    + geom_segment(data=seg_df,
                   mapping=aes(x='x', xend='xend', y='y', yend='yend'),
                   inherit_aes=False, size=0.5, color='black')
    + geom_text(data=txt_df,
                mapping=aes(x='x', y='y', label='label'),
                inherit_aes=False, size=6, ha='center', va='bottom')
    + scale_x_discrete(limits=group_order, labels=group_labels)
    + scale_fill_manual(values=group_colors)
    + coord_cartesian(ylim=(None, y_lim))
    + labs(x='', y='Relative SASA per residue')
    + theme_classic()
    + theme(axis_text_x=element_text(size=9))
)

os.makedirs('figure_panels', exist_ok=True)
p.save(f'figure_panels/{OUTPUT_PREFIX}_sasa_boxplot.pdf', bbox_inches='tight')
p.draw()
print(f'Saved: figure_panels/{OUTPUT_PREFIX}_sasa_boxplot.pdf')


Group sizes (residues):
lip_group
not detected         118
detected, non-sig     52
significant           49
correlating           54


/Users/xhernandez/miniconda3/envs/lipms_structure/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6.4 x 4.8 in image.
/Users/xhernandez/miniconda3/envs/lipms_structure/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figure_panels/STM1_sasa_boxplot.pdf


Saved: figure_panels/STM1_sasa_boxplot.pdf


## 7. Visualization: Peptide Level

All atoms shown as spacefill (grey). Significant peptide residues highlighted by category.

In [230]:
# Load structure string (done once, reused for both views)
print(f'Loading CIF structure ({file_size_mb:.1f} MB)...')
with open(cif_path) as f:
    cif_str = f.read()
print('Done.')

Loading CIF structure (0.3 MB)...
Done.


In [231]:
import re as _re

def _classify_chains(cif_text):
    '''Identify all RNA/DNA polymer chains in a mmCIF file.

    Covers rRNA, tRNA, mRNA, snRNA, DNA, and any other nucleic acid chain.

    Step 1: find RNA/DNA entity IDs from _entity description + _entity_poly type.
    Step 2: map entity_id -> label_asym_id via _struct_asym.
    Step 3: map label_asym_id -> auth_asym_id via _atom_site.

    Returns:
        rna_chains (set): auth_asym_ids for all RNA/DNA polymer chains.
    '''
    cut  = cif_text.find('\n_atom_site.')
    if cut < 0:
        cut = cif_text.find('\nloop_\n_atom_site.')
    meta = cif_text[:cut] if cut > 0 else cif_text[:500_000]

    # ── Step 1: entity IDs for RNA/DNA ───────────────────────────────────────
    rna_eids = set()

    # Unquoted entity type column (e.g. "1  1  polymer  tRNA")
    for m in _re.finditer(
            r'^\s*(\d+)\s+\S+\s+\S+\s+(?:tRNA|rRNA|mRNA|snRNA|snoRNA|RNA|DNA)\b',
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Single-quoted descriptions
    for m in _re.finditer(
            r"^\s*(\d+)\s+\S+\s+\S+\s+'([^']*(?:RNA|DNA|transfer|ribosomal|messenger)[^']*)'",
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Double-quoted descriptions
    for m in _re.finditer(
            r'^\s*(\d+)\s+\S+\s+\S+\s+"([^"]*(?:RNA|DNA|transfer|ribosomal|messenger)[^"]*)"',
            meta, _re.MULTILINE | _re.IGNORECASE):
        rna_eids.add(m.group(1))

    # Also check _entity_poly.type for polyribonucleotide / polydeoxyribonucleotide
    ep_idx = meta.find('_entity_poly.')
    if ep_idx >= 0:
        blk_start = meta.rfind('loop_', 0, ep_idx)
        blk_end   = meta.find('\nloop_', ep_idx + 1)
        blk_end   = blk_end if blk_end > 0 else len(meta)
        blk       = meta[blk_start:blk_end]
        ep_hdrs   = _re.findall(r'_entity_poly\.(\S+)', blk)
        if 'entity_id' in ep_hdrs and 'type' in ep_hdrs:
            eid_col  = ep_hdrs.index('entity_id')
            type_col = ep_hdrs.index('type')
            last_hdr = f'_entity_poly.{ep_hdrs[-1]}'
            ds       = blk.find(last_hdr) + len(last_hdr)
            for line in blk[ds:].split('\n'):
                parts = line.split()
                if len(parts) > max(eid_col, type_col):
                    ptype = parts[type_col].lower()
                    if 'ribo' in ptype or 'deoxyribo' in ptype or 'dna' in ptype:
                        rna_eids.add(parts[eid_col])

    print(f'  RNA/DNA entity IDs: {rna_eids}')
    if not rna_eids:
        return set()

    # ── Step 2: label_asym_id for RNA entities ───────────────────────────────
    label_ids  = set()
    search_pos = 0
    while True:
        sa_idx = meta.find('_struct_asym.id', search_pos)
        if sa_idx < 0:
            break
        blk_start = meta.rfind('loop_', 0, sa_idx)
        blk_end   = meta.find('\nloop_', sa_idx)
        blk_end   = blk_end if blk_end > 0 else len(meta)
        blk       = meta[blk_start:blk_end]
        headers   = _re.findall(r'_struct_asym\.(\S+)', blk)
        try:
            id_col  = headers.index('id')
            eid_col = headers.index('entity_id')
        except ValueError:
            search_pos = sa_idx + 1
            continue
        last_hdr   = f'_struct_asym.{headers[-1]}'
        data_start = blk.find(last_hdr) + len(last_hdr)
        for line in blk[data_start:].split('\n'):
            parts = line.split()
            if len(parts) > max(id_col, eid_col) and parts[eid_col] in rna_eids:
                label_ids.add(parts[id_col])
        search_pos = sa_idx + 1

    print(f'  RNA/DNA label_asym_ids: {label_ids}')
    if not label_ids:
        return set()

    # ── Step 3: map label_asym_id -> auth_asym_id via _atom_site ─────────────
    atom_start = cif_text.find('\nloop_\n_atom_site.')
    if atom_start < 0:
        atom_start = cif_text.find('\n_atom_site.')
    hdr_block  = cif_text[atom_start:atom_start + 5000]
    hdr_names  = _re.findall(r'_atom_site\.(\S+)', hdr_block)
    try:
        label_col = hdr_names.index('label_asym_id')
        auth_col  = hdr_names.index('auth_asym_id')
    except ValueError:
        print('  WARNING: could not find label_asym_id/auth_asym_id columns; returning label IDs')
        return label_ids  # fallback

    last_hdr_str  = f'_atom_site.{hdr_names[-1]}'
    data_off      = cif_text.find(last_hdr_str, atom_start) + len(last_hdr_str)
    label_to_auth = {}
    remaining     = label_ids.copy()
    for line in cif_text[data_off:].split('\n'):
        if not remaining:
            break
        parts = line.split()
        if len(parts) <= max(label_col, auth_col):
            continue
        lbl = parts[label_col]
        if lbl in remaining:
            label_to_auth[lbl] = parts[auth_col]
            remaining.discard(lbl)

    auth_ids = set(label_to_auth.values())
    print(f'  RNA/DNA auth_asym_ids (for py3Dmol): {sorted(auth_ids)}')
    return auth_ids


rna_chains = _classify_chains(cif_str)
print(f'Detected RNA/DNA chains: {sorted(rna_chains) if rna_chains else "none found"}')


  RNA/DNA entity IDs: set()
Detected RNA/DNA chains: none found


In [232]:
view1 = py3Dmol.view(width=950, height=620)
view1.addModel(cif_str, 'cif')

# ── 1. Base style: semi-transparent cartoon for everything ───────────────────
# RNA/DNA chains will show as ribbons; protein atoms will be overridden below.
view1.setStyle({}, {'cartoon': {'color': '#444444', 'opacity': 0.5}})

# ── 2. Remove cartoon from protein chains (replaced by VDW surfaces below) ───
all_prot_chains = {cm['chain_id'] for uid in sifts for cm in sifts[uid]} - rna_chains
for chain in all_prot_chains:
    view1.setStyle({'chain': chain}, {})

# ── 3. Small molecules: black sticks (non-water HETATM) ──────────────────────
view1.setStyle({'hetflag': True}, {'stick': {'color': 'black', 'radius': 0.25}})
view1.setStyle({'hetflag': True, 'resn': 'HOH'}, {})   # hide water

# ── 4. All RNA/DNA chains: opaque black cartoon ───────────────────────────────
for chain in rna_chains:
    view1.setStyle({'chain': chain}, {'cartoon': {'color': '#888888', 'opacity': 0.2}})

# ── 5. Group significant/correlating residues by chain ───────────────────────
corr_by_chain = defaultdict(set)
for chain, res in corr_res_set:
    corr_by_chain[chain].add(res)

sig_only_by_chain = defaultdict(set)
for chain, res in sig_res_set:
    if res not in corr_by_chain[chain]:
        sig_only_by_chain[chain].add(res)

# ── 6. Build per-chain full residue sets from SIFTS ──────────────────────────
chain_all_residues_sifts = defaultdict(set)
for uid, chain_maps in sifts.items():
    for cm in chain_maps:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            chain_all_residues_sifts[cm['chain_id']].add(unp_res + offset)

# ── 7. Classify protein chains: focus (UNIPROT_FILTER) vs background ─────────
# If UNIPROT_FILTER is set: chains for those proteins → COLOR_BASE (focus),
#   all others → COLOR_FILTERED (or hidden if COLOR_FILTERED is None).
# If UNIPROT_FILTER is None: all chains are focus → COLOR_BASE.
if UNIPROT_FILTER is not None:
    focus_chains      = {cm['chain_id'] for uid in UNIPROT_FILTER
                         for cm in sifts.get(uid, [])}
    background_chains = all_prot_chains - focus_chains
else:
    focus_chains      = all_prot_chains
    background_chains = set()

# ── 8. Render background chains ──────────────────────────────────────────────
if COLOR_FILTERED is not None:
    for chain in background_chains:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': COLOR_FILTERED},
                         {'chain': chain})

# ── 9. Render focus chain surfaces (COLOR_BASE + sig/corr highlights) ────────
for chain in focus_chains:
    colored = sig_only_by_chain[chain] | corr_by_chain[chain]

    if colored and chain in chain_all_residues_sifts:
        base_residues = sorted(chain_all_residues_sifts[chain] - colored)
        if base_residues:
            view1.addSurface(py3Dmol.VDW,
                             {'opacity': 0.8, 'color': COLOR_BASE},
                             {'chain': chain, 'resi': base_residues})
    else:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': COLOR_BASE},
                         {'chain': chain})

    # Significant non-correlating residues
    if sig_only_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_SIG},
                         {'chain': chain, 'resi': sorted(sig_only_by_chain[chain])})

    # Correlating residues
    if corr_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_CORR},
                         {'chain': chain, 'resi': sorted(corr_by_chain[chain])})

view1.zoomTo()
view1.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
os.makedirs('plots', exist_ok=True)
out_path1 = f'plots/{OUTPUT_PREFIX}_peptide_level.html'
view1.write_html(out_path1)
print(f'Saved: {out_path1}')

Saved: plots/SES1_peptide_level.html
